# UNIDAD 7: Inferencia Estadística y Estimación de Parámetros
> **Asignatura: Probabilidad y Estadística Inferencial**
> **UCEMICH — Ingeniería en IA y Nanotecnología**
> **Autor y Profesor: Mtro. Luis José Yudico Anaya**

## 1. Fundamentación Teórica de la Inferencia Estadística

La **Inferencia Estadística** comprende el conjunto de métodos que permiten deducir las propiedades de una población completa a partir del análisis de una muestra representativa.

### 1.1 IMPORTANDO MÓDULOS

In [ ]:
import scipy.stats as stats
from scipy import optimize
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
sns.set(style="whitegrid")


Otros estilos de `sns.set()`:

Seaborn también ofrece varios otros estilos de gráficos que puedes usar:

* `"darkgrid"`: Fondo oscuro con cuadrícula en líneas claras (ideal para gráficos con contraste alto).
* `"white"`: Fondo blanco sin cuadrícula (minimalista).
* `"ticks"`: Estilo con cuadrícula ligera y marcas de los ejes más prominentes.

### 1.2 Revisión de estadística y probabilidad.

Comenzamos con una breve revisión de la estadística, para introducir algunos de los conceptos clave y la notación que utilizamos en este y los siguientes capítulos.

La estadística se ocupa de la recopilación y el análisis de datos con el propósito de obtener información, sacar conclusiones y apoyar la toma de decisiones. Los métodos estadísticos son necesarios cuando tenemos información incompleta sobre un fenómeno. Generalmente, tenemos información incompleta porque no podemos recopilar datos de todos los miembros de una población o si existe incertidumbre en las observaciones que realizamos (por ejemplo, debido al ruido de medición). Cuando no podemos encuestar a toda una población, podemos estudiar una muestra elegida aleatoriamente, y utilizar métodos estadísticos para calcular estadísticas descriptivas (parámetros como la media $ \mu $ y las varianzas $ \sigma^2 $) con el fin de hacer inferencias sobre las propiedades de toda la población (también llamada espacio muestral) de manera sistemática y con riesgos controlados de error.

Los métodos estadísticos se construyen sobre la base proporcionada por la teoría de la probabilidad, con la cual podemos modelar la incertidumbre y la información incompleta utilizando variables aleatorias probabilísticas.

Por ejemplo, con muestras seleccionadas aleatoriamente de una población, podemos esperar obtener muestras representativas cuyas propiedades pueden ser usadas para inferir propiedades de toda la población.

En la teoría de la probabilidad, a cada posible resultado de una observación se le asigna una probabilidad, y la probabilidad de todos los resultados posibles constituye la distribución de probabilidad. Dada la distribución de probabilidad, podemos calcular las propiedades de la población, como su media $ \mu $ y su varianza $ \sigma^2 $, pero para muestras seleccionadas aleatoriamente, solo conocemos los resultados esperados o promedio.

En el análisis estadístico es importante **distinguir entre las estadísticas de población y las de muestra**. Aquí denotamos los parámetros de la población con símbolos griegos y los parámetros de una muestra con el símbolo correspondiente de la población, añadiendo el subíndice $x$ (o el símbolo que se utiliza para representar la muestra). Por ejemplo, la media y la varianza de una población se denotan como $ \mu $ y $ \sigma^2 $, mientras que la media y la varianza de una muestra $ x $ se denotan como $ \mu_x $ y $ s_x^2 $.

Además, denotamos las variables que representan una población (variables aleatorias) con letras mayúsculas, por ejemplo, $X$, y un conjunto de elementos de muestra se denota con una letra minúscula, por ejemplo, $x$. Una barra sobre un símbolo denota el promedio o la media, así que:

$$
\overline{X} = \frac{1}{N} \sum_{i=1}^N x_i
$$

y

$$
\overline{x} = \frac{1}{n} \sum_{i=1}^n x_i
$$

donde $N$ es el número de elementos en la población $X$ y $n$ es el número de elementos en la muestra $x$. La única diferencia entre estas dos expresiones es el número de elementos en la suma ($N \geq n$).

La situación es un poco más compleja para la varianza: la varianza de la población es la media de la distancia al cuadrado de la media,

$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2
$$

y la varianza de la muestra correspondiente es

$$
s_x^2 = \frac{1}{n-1} \sum_{i=1}^n (x_i - \mu_x)^2
$$

En esta última expresión, hemos reemplazado la media de la población $ \mu $ por la media de la muestra $ \mu_x $ y dividido la suma por $n - 1$ en lugar de $n$. La razón de esto es que se ha eliminado un grado de libertad del conjunto de la muestra al calcular la media de la muestra $ \mu_x $, por lo que, al calcular la varianza de la muestra, solo quedan $n - 1$ grados de libertad. En consecuencia, la forma de calcular la varianza para una población y para una muestra es ligeramente diferente. Esto se refleja en las funciones que podemos utilizar para calcular estas estadísticas en Python.

Podemos calcular estadísticas descriptivas para datos utilizando funciones NumPy o los métodos `ndarray` correspondientes. Por ejemplo, a
calcular la media y la mediana de un conjunto de datos, podemos usar las funciones NumPy media y mediana:

In [ ]:
x = np.array([3.5, 1.1, 3.2, 2.8, 6.7, 4.4, 0.9, 2.2])


In [ ]:
np.mean(x)


In [ ]:
np.median(x)


De manera similar, podemos usar funciones `min` y `max` o métodos `ndarray` para calcular los valores mínimo y máximo en la matriz:

In [ ]:
x.min(), x.max()


Para calcular la varianza y la desviación estándar de un conjunto de datos, utilizamos `var`y métodos estándar. Por defecto, la fórmula para la varianza de la población y el estándar.
Se utiliza la desviación (es decir, se supone que el conjunto de datos es toda la población).

In [ ]:
x.var()


In [ ]:
x.std()


Sin embargo, para cambiar este comportamiento, podemos usar el argumento `ddof` (grados delta de libertad). El denominador en la expresión de la varianza es el número de elementos en la matriz menos `ddof`, por lo que para calcular la estimación insesgada de la varianza y desviación estándar de una muestra, debemos establecer` ddof=1`:

In [ ]:
x.var(ddof=1)


In [ ]:
x.std(ddof=1)


### 1.3 Números Casi- Aleatorios

La biblioteca estándar de Python contiene el módulo `random`, que proporciona funciones para generar números aleatorios individuales con algunas distribuciones básicas. El módulo `random` en el módulo NumPy proporciona una funcionalidad similar, pero ofrece funciones que generan arreglos de NumPy con números aleatorios, y tiene soporte para una selección más amplia de distribuciones de probabilidad. Los arreglos con números aleatorios son a menudo prácticos para fines computacionales, por lo que aquí nos centramos en el módulo `random` de NumPy, y más adelante también en las funciones y clases de mayor nivel en `scipy.stats`, que se basan en NumPy y lo extienden.

Anteriormente en este libro, ya hemos utilizado `np.random.rand`, que genera números de punto flotante distribuidos uniformemente en el intervalo semiabierto $[0, 1)$ (es decir, $0.0$ es un resultado posible, pero $1.0$ no lo es).

Además de esta función, el módulo `np.random` también contiene una gran colección de otras funciones para generar números aleatorios que cubren diferentes intervalos, tienen diferentes distribuciones y toman valores de diferentes tipos (por ejemplo, números de punto flotante y enteros). Por ejemplo:
* la función `randn` produce números aleatorios distribuidos según la distribución normal estándar (la distribución normal con media $0$ y desviación estándar $1$),
* y la función `randint` genera enteros distribuidos uniformemente entre un valor bajo (inclusive) y un valor alto (exclusive).

Cuando las funciones `rand` y `randn` se llaman sin argumentos, producen un único número aleatorio:

In [ ]:
np.random.rand()


In [ ]:
np.random.randn()


Sin embargo, pasar la forma del arreglo como argumentos a estas funciones produce arreglos de números aleatorios. Por ejemplo, aquí generamos un vector de longitud $5$ usando `rand` pasando un solo argumento $5$ y un arreglo de $2 \times 4$ usando `randn` pasando los argumentos $2$ y $4$ (los arreglos de mayor dimensión se generan pasando la longitud de cada dimensión como argumentos):

In [ ]:
np.random.rand(5)


In [ ]:
np.random.randn(2, 4)


Para generar enteros aleatorios utilizando `randint` (ver también `random_integers`), necesitamos proporcionar ya sea el límite superior para los números aleatorios (en cuyo caso el límite inferior es implícitamente cero) o ambos límites, el inferior y el superior. El tamaño del arreglo generado se especifica utilizando el argumento clave `size`, y puede ser un entero o una tupla que especifica la forma de un arreglo multidimensional:

In [ ]:
np.random.randint(10, size=10)


In [ ]:
np.random.randint(low=10, high=20, size=(2, 10))


Cabe destacar que la función `randint` genera enteros aleatorios en el intervalo semiabierto $[low, high)$.

Para demostrar que los números aleatorios producidos por `rand`, `randn` y `randint` están distribuidos de manera diferente, podemos trazar los histogramas de, por ejemplo, $10000$ números aleatorios producidos por cada función.
El resultado se muestra en la Figura 13-1. Observamos que las distribuciones de `rand` y `randint` parecen ser uniformes pero tienen rangos y tipos diferentes, mientras que la distribución de los números producidos por `randn` se asemeja a una curva gaussiana centrada en cero, como se esperaba.

**GRÁFICO CON MATPLOTLIB**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 3))
...: axes[0].hist(np.random.rand(10000))
...: axes[0].set_title("rand")
...: axes[1].hist(np.random.randn(10000))
...: axes[1].set_title("randn")
...: axes[2].hist(np.random.randint(low=1, high=10, size=10000),

bins=9, align='left')

...: axes[2].set_title("randint(low=1, high=10)")

plt.show()


**GRÁFICO CON SEABORN**

In [ ]:
from pickle import TRUE
## Crear la figura con 3 subgráficos
fig, axes = plt.subplots(1, 3, figsize=(18, 3))

## Histograma usando Seaborn
sns.histplot(np.random.rand(10000), ax=axes[0], kde=True)
axes[0].set_title("rand")

sns.histplot(np.random.randn(10000), ax=axes[1], kde=False)
axes[1].set_title("randn")

sns.histplot(np.random.randint(low=1, high=10, size=10000), ax=axes[2], bins=9, kde=False)
axes[2].set_title("randint(low=1, high=10)")

## Mostrar el gráfico
plt.show()


En el análisis estadístico, a menudo es necesario generar una lista única de enteros. Esto corresponde a realizar un muestreo (selección aleatoria) de elementos de un conjunto (población) sin reposición (de modo que no obtenemos el mismo elemento dos veces). Desde el módulo aleatorio de NumPy, podemos utilizar la función `choice` para generar este tipo de números aleatorios. Como primer argumento, podemos proporcionar una lista (o arreglo) con los valores de la población o un número entero que corresponde al número de elementos en la población. Como segundo argumento, damos el número de valores que se van a muestrear. Si los valores deben ser muestreados con o sin reposición se puede especificar utilizando el argumento clave `replace`, que toma los valores booleanos $True$ o $False$.

Por ejemplo, para muestrear cinco elementos únicos (sin reposición) del conjunto de enteros entre $0$ (inclusive) y $10$ (exclusive), podemos usar:

In [ ]:
np.random.choice(10, 5, replace=False)


Cuando se trabaja con generación de números aleatorios, puede ser útil "sembrar" el generador de números aleatorios. La semilla es un número que inicializa el generador de números aleatorios a un estado específico, de modo que, una vez que ha sido sembrado con un número específico, siempre genera la misma secuencia de números aleatorios. Esto puede ser útil cuando se realiza pruebas y para reproducir resultados previos, y ocasionalmente en aplicaciones que requieren volver a sembrar el generador de números aleatorios (por ejemplo, después de haber bifurcado un proceso).

Para sembrar el generador de números aleatorios en NumPy, podemos usar la función `seed`, que toma un número entero como argumento:

In [ ]:
np.random.seed(123456789)
np.random.rand()


In [ ]:
np.random.rand()


Cabe destacar que, después de haber sembrado el generador de números aleatorios con un número específico, aquí $123456789$, las siguientes llamadas a los generadores de números aleatorios siempre producen los mismos resultados. La semilla del generador de números aleatorios es un estado global del módulo `np.random`.

Un nivel más fino de control sobre el estado del generador de números aleatorios se puede lograr utilizando la clase `RandomState`, que opcionalmente toma un número entero como argumento para su inicializador. El objeto `RandomState` mantiene un registro del estado del generador de números aleatorios y permite mantener varios generadores de números aleatorios independientes en el mismo programa (lo cual puede ser útil, por ejemplo, cuando se trabaja con aplicaciones multihilo).

Una vez que se ha creado un objeto `RandomState`, podemos utilizar los métodos de este objeto para generar números aleatorios. La clase `RandomState` tiene métodos que corresponden a las funciones disponibles en el módulo `np.random`. Por ejemplo, podemos usar el método `randn` de la clase `RandomState` para generar números aleatorios distribuidos según la normal estándar:

In [ ]:
prng = np.random.RandomState(123456789)
prng.randn(2, 4)


De manera similar, existen métodos como `rand`, `randint`, `rand_integers` y `choice`, que también corresponden a las funciones en el módulo `np.random` con el mismo nombre. Se considera una buena práctica de programación utilizar una instancia de `RandomState` en lugar de usar directamente las funciones en el módulo `np.random`, porque evita depender de una variable de estado global y mejora el aislamiento del código. Esto es una consideración importante al desarrollar funciones de biblioteca que usan números aleatorios, pero quizás sea menos relevante en aplicaciones y cálculos más pequeños.

### 1.4 Simulando Distribuciones

Además de las distribuciones fundamentales de números aleatorios que hemos visto hasta ahora (distribuciones uniformes discretas y continuas, `randint` y `rand`, y la distribución normal estándar, `randn`), también existen funciones y métodos de `RandomState` para una gran cantidad de distribuciones de probabilidad que ocurren en estadística. Por mencionar solo algunas, están la distribución continua $ \chi^2 $ (chi-cuadrado, `chisquare`), la distribución de Student $t$ (distribución `standard_t`), y la distribución $F$ (distribución `f`):

In [ ]:
prng.chisquare(1, size=(2, 2))


In [ ]:
prng.standard_t(1, size=(2, 3))


In [ ]:
prng.f(5, 2, size=(2, 4))


y la distribución binomial discreta (binomial) y la distribución de Poisson (poisson):

In [ ]:
prng.binomial(10, 0.5, size=10)


In [ ]:
prng.poisson(5, size=10)


Lista de distribuciones disponibles dentro de la clase `RandomState` en NumPy:

### 1.5 Distribuciones continuas:

- **beta(a, b, size=None)**: Distribución Beta.
- **chisquare(df, size=None)**: Distribución chi-cuadrado.
- **dirichlet(alpha, size=None)**: Distribución Dirichlet.
- **exponential(scale=1.0, size=None)**: Distribución Exponencial.
- **f(dfnum, dfden, size=None)**: Distribución $F$.
- **gamma(shape, scale=1.0, size=None)**: Distribución Gamma.
- **gumbel(loc=0.0, scale=1.0, size=None)**: Distribución Gumbel.
- **laplace(loc=0.0, scale=1.0, size=None)**: Distribución Laplace.
- **logistic(loc=0.0, scale=1.0, size=None)**: Distribución Logística.
- **lognormal(mean=0.0, sigma=1.0, size=None)**: Distribución Log-normal.
- **multivariate_normal(mean, cov, size=None, check_valid='warn', tol=1e-8)**: Distribución normal multivariante.
- **noncentral_chisquare(df, nonc, size=None)**: Distribución chi-cuadrado no central.
- **noncentral_f(dfnum, dfden, nonc, size=None)**: Distribución $F$ no central.
- **normal(loc=0.0, scale=1.0, size=None)**: Distribución Normal (Gaussiana).
- **pareto(a, size=None)**: Distribución Pareto II o Lomax.
- **power(a, size=None)**: Distribución de Potencia.
- **rayleigh(scale=1.0, size=None)**: Distribución Rayleigh.
- **standard_cauchy(size=None)**: Distribución de Cauchy estándar.
- **standard_exponential(size=None)**: Distribución exponencial estándar.
- **standard_gamma(shape, size=None)**: Distribución Gamma estándar.
- **standard_normal(size=None)**: Distribución normal estándar.
- **standard_t(df, size=None)**: Distribución t de Student estándar.
- **triangular(left, mode, right, size=None)**: Distribución Triangular.
- **uniform(low=0.0, high=1.0, size=None)**: Distribución Uniforme.
- **vonmises(mu, kappa, size=None)**: Distribución Von Mises.
- **wald(mean, scale, size=None)**: Distribución Wald.
- **weibull(a, size=None)**: Distribución Weibull.

### 1.6 Distribuciones discretas:

- **binomial(n, p, size=None)**: Distribución Binomial.
- **geometric(p, size=None)**: Distribución Geométrica.
- **hypergeometric(ngood, nbad, nsample, size=None)**: Distribución Hipergeométrica.
- **logseries(p, size=None)**: Distribución Logarítmica.
- **multinomial(n, pvals, size=None)**: Distribución Multinomial.
- **negative_binomial(n, p, size=None)**: Distribución Binomial negativa.
- **poisson(lam=1.0, size=None)**: Distribución Poisson.
- **zipf(a, size=None)**: Distribución Zipf.

[random — Generate pseudo-random numbers](https://docs.python.org/es/3/library/random.html)

https://docs.python.org/es/3/library/random.html

Para obtener una lista completa de las funciones de distribución disponibles, consulta las docstrings del módulo `np.random`, usa `help(np.random)` y la clase `RandomState`. Aunque es posible usar las funciones de `np.random` y los métodos de `RandomState` para extraer números aleatorios de muchas distribuciones estadísticas diferentes, cuando se trabaja con distribuciones, existe una interfaz de más alto nivel en el módulo `scipy.stats` que combina el muestreo de números aleatorios con muchas otras funciones convenientes para distribuciones de probabilidad. En la siguiente sección, exploramos esto con más detalle.

In [ ]:
help(np.random)


In [ ]:
import scipy as sc
from scipy import stats


In [ ]:
help(stats)


### 1.7 Variables Aleatorias y Distribuciones

En la teoría de la probabilidad, el conjunto de posibles resultados de un proceso aleatorio se llama **espacio muestral**. A cada elemento del espacio muestral (es decir, un resultado de un experimento o una observación) se le puede asignar una probabilidad, y las probabilidades de todos los posibles resultados definen la **distribución de probabilidad**.

Una **variable aleatoria** es una asignación del espacio muestral a los números reales o a los enteros. Por ejemplo, los posibles resultados de un lanzamiento de moneda son cara y cruz, por lo que el espacio muestral es $\{\text{cara}, \text{cruz}\}$, y una posible variable aleatoria toma el valor $0$ para cara y $1$ para cruz.

En general, hay muchas maneras de definir variables aleatorias para los posibles resultados de un proceso aleatorio dado. Las variables aleatorias son una representación independiente del problema de un proceso aleatorio. Es más fácil trabajar con variables aleatorias porque están descritas por números en lugar de resultados de espacios muestrales específicos del problema.

Un paso común en la resolución de problemas estadísticos es, por lo tanto, mapear los resultados a valores numéricos y determinar la distribución de probabilidad de esos valores.

En consecuencia, una **variable aleatoria** se caracteriza por sus posibles valores y su **distribución de probabilidad**, que asigna una probabilidad a cada valor posible. Cada observación de la variable aleatoria resulta en un número aleatorio, y la distribución de los valores observados se describe mediante la distribución de probabilidad. Existen dos tipos principales de distribuciones: **discretas** y **continuas**, que corresponden a valores enteros y valores reales, respectivamente.

Al trabajar con estadísticas, tratar con variables aleatorias es de gran importancia, y en la práctica esto a menudo significa trabajar con distribuciones de probabilidad. El módulo `scipy.stats` proporciona clases para representar variables aleatorias con una gran cantidad de distribuciones de probabilidad. Existen dos clases base para variables aleatorias discretas y continuas: `rv_discrete` y `rv_continuous`.

Estas clases no se utilizan directamente, sino como clases base para variables aleatorias con distribuciones específicas, y definen una interfaz común para todas las clases de variables aleatorias en `scipy.stats`. Un resumen de los métodos seleccionados para variables aleatorias discretas y continuas se presenta en la Tabla 13-1.

### 1.8 Tabla 13-1. Métodos seleccionados para variables aleatorias discretas y continuas en el módulo `scipy.stats`

| Métodos     | Descripción |
|-------------|-------------|
| **pdf/pmf** | Función de distribución de probabilidad (para variables continuas) o función de masa de probabilidad (para variables discretas). |
| **cdf**     | Función de distribución acumulada. |
| **sf**      | Función de supervivencia ($1 - \text{cdf}$). |
| **ppf**     | Función percentil (inversa de la cdf). |
| **moment**  | Momentos no centrales de orden $n$. |
| **stats**   | Estadísticas de la distribución (típicamente la media y la varianza, a veces estadísticas adicionales). |
| **fit**     | Ajuste de la distribución a los datos utilizando una optimización de máxima verosimilitud numérica (para distribuciones continuas). |
| **expect**  | Valor de la expectativa de una función con respecto a la distribución. |
| **interval**| Los puntos finales del intervalo que contiene un porcentaje dado de la distribución (intervalo de confianza). |
| **rvs**     | Muestras de la variable aleatoria. Toma como argumento el tamaño del array resultante de las muestras. |
| **mean, median, std, var** | Estadísticas descriptivas: media, mediana, desviación estándar y varianza de la distribución. |

### 1.9 Métodos adicionales para variables aleatorias discretas:

| Métodos     | Descripción |
|-------------|-------------|
| **entropy** | Calcula la entropía de la distribución. |
| **support** | Devuelve una tupla que contiene el límite inferior y superior del soporte de la distribución. |

### 1.10 Métodos adicionales para variables aleatorias continuas:

| Métodos     | Descripción |
|-------------|-------------|
| **isf**     | Función de cuantil inversa de supervivencia (inversa de la sf). |
| **logpdf/logpmf** | Logaritmo de la función de densidad de probabilidad o función de masa de probabilidad. |
| **logcdf**  | Logaritmo de la función de distribución acumulada. |
| **logsf**   | Logaritmo de la función de supervivencia. |

Además de estos métodos, también hay varios otros métodos específicos para ciertas distribuciones. Para obtener una lista completa de todos los métodos disponibles para una distribución específica, consulta la documentación de `scipy.stats` usando `help(stats.<distribution_name>)`.

In [ ]:
help(stats.norm)


**Referencia**:

[Statistical functions (scipy.stats)](https://docs.scipy.org/doc/scipy/reference/stats.html)

https://docs.scipy.org/doc/scipy/reference/stats.html

Existen una gran cantidad de clases para variables aleatorias discretas y continuas en el módulo `scipy.stats`. En el momento de escribir este texto, hay clases para 13 distribuciones discretas y 98 distribuciones continuas, que incluyen las distribuciones más comúnmente encontradas (y muchas menos comunes). Para una referencia completa, consulta la docstring del módulo `stats`: `help(stats)`.

A continuación, exploramos algunas de las distribuciones más comunes, pero el uso de todas las demás distribuciones sigue el mismo patrón.

Las clases de variables aleatorias en el módulo `scipy.stats` tienen varios casos de uso. Son representaciones de la distribución, que pueden utilizarse para calcular estadísticas descriptivas y para graficar, y también pueden usarse para generar números aleatorios siguiendo la distribución dada, utilizando el método `rvs` (muestra de variable aleatoria). Este último caso de uso es similar al que utilizamos con el módulo `np.random` anteriormente en este capítulo.

Como demostración de cómo utilizar las clases de variables aleatorias en `scipy.stats`, consideremos el siguiente ejemplo en el que creamos una variable aleatoria distribuida normalmente con una media de $1.0$ y una desviación estándar de $0.5$:

In [ ]:
X = stats.norm(1, 0.5)


Ahora $X$ es un objeto que representa una variable aleatoria y podemos calcular
estadística descriptiva de esta variable aleatoria utilizando, por ejemplo, la media, mediana, estándar y métodos var:

In [ ]:
X.mean()


In [ ]:
X.median()


In [ ]:
X.std()


In [ ]:
X.var()


Los momentos no centrales de orden arbitrario se pueden calcular con el método del momento:

In [ ]:
[X.moment(n) for n in range(5)]


**NOTA**

`X.moment(n)` calcula momentos crudos, no momentos centrales.

El método `.moment(n)` de un objeto de distribución de `scipy.stats` calcula los momentos crudos o no centrales. Los momentos crudos se definen como:

$$
E[X^n] \quad \text{ Valor esperado de } X \text{ elevado a la potencia de } n
$$

Por otro lado, los momentos centrales se definen como:

$$
E[(X - \mu)^n] \quad \text{ Valor esperado de } (X - \mu) \text{ elevado a la potencia de } n
$$

**¿Por qué el primer momento crudo no es 0?**

El primer momento crudo ($n=1$) es simplemente el valor esperado (media) de la distribución, que en tu caso es $1.0$. No es el momento central, que se calcularía como:

$$
E[(X - \mu)^1]
$$

Y que, en efecto, sería igual a 0.

En teoría de probabilidad y estadística, los momentos centrales se utilizan para caracterizar la forma de una distribución de probabilidad. Se calculan como el valor esperado de la diferencia entre una variable aleatoria y su media, elevada a una determinada potencia. Los momentos centrales proporcionan información sobre el centro, la dispersión, la asimetría y la curtosis de la distribución.

### 1.11 Momentos Centrales

Para una variable aleatoria $X$ con una media denotada por $\mu$, el momento central de orden $n$ se define como:

$$
E\left[(X - \mu)^n\right]
$$

donde $E[\cdot]$ denota el operador de valor esperado.

### 1.12 Entendiendo los Momentos Centrales

- **Momento Central de orden 0**: Siempre es igual a 1.
- **Momento Central de orden 1**: Siempre es igual a 0 (ya que es la desviación esperada de la media).
- **Momento Central de orden 2**: Este es la varianza ($\sigma^2$), que mide la dispersión o la extensión de la distribución.
- **Momento Central de orden 3**: Se usa para calcular la asimetría (skewness), que indica la falta de simetría de la distribución.
- **Momento Central de orden 4**: Se usa para calcular la curtosis (kurtosis), que mide la "apuntamiento" de la distribución (qué tan concentrada está la probabilidad alrededor de la media y en las colas).

### 1.13 El código:

El fragmento de código:

$$
[X.moment(n) \text{ for } n \text{ en el rango de } 5]
$$

está calculando los primeros cinco momentos no centrales (momentos crudos) de la distribución normal representada por la variable $X$. El método `.moment(n)` del objeto `scipy.stats.norm` te da el $n$-ésimo momento crudo, no el momento central.

### 1.14 Diferencia clave

Los momentos crudos usan $E[X^n]$ en lugar de $E[(X - \mu)^n]$ en sus cálculos.

Para calcular momentos centrales en tu código, primero deberías calcular la media $X.\text{mean}()$ y luego aplicar manualmente la fórmula del momento central usando las funciones de NumPy.

In [ ]:
X = stats.norm(1, 0.5)


In [ ]:
mean = X.mean()


In [ ]:
data = X.rvs(size=1000)  # Generate 1000 samples


In [ ]:
import numpy as np

## 1erd central moment (variance)
first_central_moment = np.mean((data - mean)**1)
first_central_moment


In [ ]:
## 2nd central moment (variance)
second_central_moment = np.mean((data - mean)**2)
second_central_moment


#### ¿Por qué 0.23999821698807636?

#### Varianza de la muestra:
Este valor es una estimación de la varianza basada en tu muestra de 1000 puntos de datos.

#### Varianza teórica:
La varianza teórica de tu distribución normal ($X$) es 0.25 (ya que la desviación estándar es 0.5 y la varianza se calcula como la desviación estándar al cuadrado: $\sigma^2 = 0.5^2 = 0.25$).

#### Variabilidad de muestreo:
Dado que estás utilizando una muestra finita (1000 puntos de datos) para estimar la varianza de una distribución infinita, habrá algo de variación aleatoria. Tu varianza calculada ($0.239998...$) está cerca del valor teórico pero no es exactamente la misma debido a esta variabilidad de muestreo.

#### Ley de los grandes números:
Si aumentaras significativamente el tamaño de tu muestra (por ejemplo, a 10,000 o 100,000), la varianza calculada probablemente se acercaría aún más a la varianza teórica de 0.25. Esto se debe a la ley de los grandes números, que establece que a medida que el tamaño de la muestra aumenta, la media y la varianza muestrales convergen hacia la verdadera media y varianza de la población.

#### En resumen:
El valor calculado de $0.239998...$ es una estimación de la varianza, y su ligera desviación del valor teórico es esperada debido a la naturaleza aleatoria del proceso de muestreo.

In [ ]:
## 3rd central moment
third_central_moment = np.mean((data - mean)**3)

## 4th central moment
fourth_central_moment = np.mean((data - mean)**4)

## etc...


Y podemos obtener una lista de estadísticas dependiente de la distribución utilizando el método `stats`.
(aquí, para una variable aleatoria distribuida normal, obtenemos la media y la varianza):

In [ ]:
X.stats()


### 1.15 Evaluación de funciones de distribución

Podemos evaluar la función de densidad de probabilidad (pdf), la función de distribución acumulada (cdf), la función de supervivencia (sf), etc., utilizando los métodos `pdf`, `cdf`, `sf`, entre otros. Todos estos métodos toman un valor o un arreglo de valores en los que evaluar la función:

In [ ]:
X.pdf([0, 1, 2])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

## Asumiendo que X ya está definido como stats.norm(1, 0.5)
x_values = [0, 1, 2]
pdf_values = X.pdf(x_values)

## Crear un rango de valores de x para la curva suave
x_range = np.linspace(X.ppf(0.01), X.ppf(0.99), 100)
pdf_range = X.pdf(x_range)

## Crear el gráfico con Seaborn
sns.lineplot(x=x_range, y=pdf_range) # Curva suave
plt.stem(x_values, pdf_values, linefmt='r-', markerfmt='ro', basefmt='k-') # Puntos específicos
plt.title('Función de Densidad de Probabilidad (PDF)')
plt.xlabel('x')
plt.ylabel('PDF(x)')
plt.grid(True)
plt.show()


In [ ]:
X.cdf([0, 1, 2, 3, 4])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

## Asumiendo que X ya está definido como stats.norm(1, 0.5)
x_values = [0, 1, 2, 3, 4]
cdf_values = X.cdf(x_values)

## Crear el gráfico con Seaborn
sns.lineplot(x=x_values, y=cdf_values, marker='o')
plt.title('Función de Distribución Acumulada (CDF)')
plt.xlabel('x')
plt.ylabel('CDF(x)')
plt.grid(True)
plt.show()


### 1.16 Método de intervalo

El método `interval` puede ser utilizado para calcular los valores inferiores y superiores de $x$ tales que un porcentaje dado de la distribución de probabilidad cae dentro del intervalo $(\text{lower}, \text{upper})$. Este método es útil para calcular intervalos de confianza y para seleccionar un rango de valores de $x$ para graficar.

In [ ]:
X.interval(0.95)


In [ ]:
X.interval(0.99)


In [ ]:
X = stats.norm(1, 0.5)  # Mean = 1, Standard Deviation = 0.5
lower_bound, upper_bound = X.interval(0.99)

x = np.linspace(X.ppf(0.001), X.ppf(0.999), 100)
plt.plot(x, X.pdf(x), 'b-', lw=2, label='PDF')
plt.fill_between(x, X.pdf(x), where=(lower_bound < x) & (x < upper_bound),
                 color='skyblue', alpha=0.6, label='99% Interval')
plt.title('Normal Distribution with 99% Interval')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.show()


### 1.17 Graficar una distribución de probabilidad

Para construir una intuición sobre las propiedades de una distribución de probabilidad, es útil graficarla, junto con la correspondiente función de probabilidad acumulada (CDF) y la función de percentiles (PPF). Para facilitar la repetición de este proceso con varias distribuciones, primero creamos una función `plot_rv_distribution` que grafica el resultado de los métodos `pdf` o `pmf`, `cdf`, `sf`, y `ppf` de los objetos de variables aleatorias del módulo `SciPy stats`, sobre un intervalo que contiene el 99.9% de la función de distribución de probabilidad. También resaltamos el área que contiene el 95% de la distribución de probabilidad utilizando el método de dibujo `fill_between`.

In [ ]:
def plot_rv_distribution(X, axes=None):
    """Plot the PDF or PMF, CDF, SF and PPF of a given random
    variable"""
    if axes is None:
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))

    x_min_999, x_max_999 = X.interval(0.999)
    x999 = np.linspace(x_min_999, x_max_999, 1000)
    x_min_95, x_max_95 = X.interval(0.95)
    x95 = np.linspace(x_min_95, x_max_95, 1000)

    if hasattr(X.dist, "pdf"):
        axes[0].plot(x999, X.pdf(x999), label="PDF")
        axes[0].fill_between(x95, X.pdf(x95), alpha=0.25)
    else:
## discrete random variables do not have a pdf method, instead we use pmf:
        x999_int = np.unique(x999.astype(int))
        axes[0].bar(x999_int, X.pmf(x999_int), label="PMF")
    axes[1].plot(x999, X.cdf(x999), label="CDF")
    axes[1].plot(x999, X.sf(x999), label="SF")
    axes[2].plot(x999, X.ppf(x999), label="PPF")

    for ax in axes:
        ax.legend()

import matplotlib.pyplot as plt
import scipy.stats as stats

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
## Assuming plot_rv_distribution is defined elsewhere
## and takes X and axes as arguments.
X = stats.norm()
plot_rv_distribution(X, axes=axes[0, :])
axes[0, 0].set_ylabel("Normal dist.")
X = stats.f(2, 50)
plot_rv_distribution(X, axes=axes[1, :])
axes[1, 0].set_ylabel("F dist.")
X = stats.poisson(5)
plot_rv_distribution(X, axes=axes[2, :])
axes[2, 0].set_ylabel("Poisson dist.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
import pandas as pd # Import pandas for creating a DataFrame

def plot_rv_distribution(X, axes=None):
    """Plot the PDF or PMF, CDF, SF and PPF of a given random variable using Seaborn"""
    if axes is None:
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))

    x_min_999, x_max_999 = X.interval(0.999)
    x999 = np.linspace(x_min_999, x_max_999, 1000)
    x_min_95, x_max_95 = X.interval(0.95)
    x95 = np.linspace(x_min_95, x_max_95, 1000)

    if hasattr(X.dist, "pdf"):
## PDF plot using Seaborn
        axes[0].plot(x999, X.pdf(x999), label="PDF")
        axes[0].fill_between(x95, X.pdf(x95), alpha=0.25)

    else:
## PMF plot using Seaborn (for Poisson)
        x999_int = np.unique(x999.astype(int))

## Create a DataFrame for Seaborn barplot (easier to work with)
        df = pd.DataFrame({'x': x999_int, 'pmf': X.pmf(x999_int)})
        sns.barplot(x='x', y='pmf', data=df, ax=axes[0], label="PMF", color="skyblue")
        axes[0].set_xticks(x999_int)

## CDF and SF plot using Seaborn
    axes[1].plot(x999, X.cdf(x999), label="CDF")
    axes[1].plot(x999, X.sf(x999), label="SF")

## PPF plot using Seaborn
    axes[2].plot(x999, X.ppf(x999), label="PPF")

    for ax in axes:
        ax.legend()

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
X = stats.norm()
plot_rv_distribution(X, axes=axes[0, :])
axes[0, 0].set_ylabel("Normal dist.")
X = stats.f(2, 50)
plot_rv_distribution(X, axes=axes[1, :])
axes[1, 0].set_ylabel("F dist.")
X = stats.poisson(5)
plot_rv_distribution(X, axes=axes[2, :])
axes[2, 0].set_ylabel("Poisson dist.")

plt.tight_layout()
plt.show()


### 1.18 Algunas funciones de distribución

Ejemplos de funciones de distribución de probabilidad (PDF), funciones de masa de probabilidad (PMF), funciones de distribución acumulada (CDF), funciones de supervivencia (SF) y funciones de punto porcentual (PPF)

A continuación se presentan ejemplos de las funciones mencionadas para tres distribuciones comunes: una distribución normal (parte superior), una distribución F (parte media) y una distribución de Poisson (parte inferior).

### 1.19 Distribución Normal

### 1.20 Función de Distribución de Probabilidad (PDF)
La función de densidad de probabilidad para una distribución normal es:

$$
f(x) = \frac{1}{\sigma \sqrt{2\pi}} e^{-\frac{(x - \mu)^2}{2\sigma^2}}
$$

donde:
- $\mu$ es la media,
- $\sigma$ es la desviación estándar.

### 1.21 Función de Distribución Acumulada (CDF)
La función de distribución acumulada para una distribución normal es:

$$
F(x) = \frac{1}{2} \left[ 1 + \text{erf}\left( \frac{x - \mu}{\sigma \sqrt{2}} \right) \right]
$$

donde $\text{erf}$ es la función de error.

### 1.22 Función de Supervivencia (SF)
La función de supervivencia es:

$$
S(x) = 1 - F(x)
$$

### 1.23 Función de Punto Porcentual (PPF)
La función de punto porcentual es la inversa de la CDF y se denota como $F^{-1}$:

$$
\text{PPF}(p) = \mu + \sigma \sqrt{2} \, \text{erf}^{-1}(2p - 1)
$$

### 1.24 Distribución F

### 1.25 Función de Distribución de Probabilidad (PDF)
La función de densidad de probabilidad para una distribución F con parámetros $d_1$ y $d_2$ es:

$$
f(x) = \frac{\sqrt{\frac{(d_1 x)^{d_1}}{d_2^{d_2} (d_1 x + d_2)^{d_1 + d_2}}}}{B\left( \frac{d_1}{2}, \frac{d_2}{2} \right)}
$$

donde $B$ es la función beta y $d_1$ y $d_2$ son los grados de libertad.

### 1.26 Función de Distribución Acumulada (CDF)
La CDF para una distribución F es:

$$
F(x; d_1, d_2) = I_{\frac{d_1 x}{d_1 x + d_2}}\left( \frac{d_1}{2}, \frac{d_2}{2} \right)
$$

donde $I$ es la función de distribución incompleta beta.

### 1.27 Distribución de Poisson

### 1.28 Función de Masa de Probabilidad (PMF)
La función de masa de probabilidad para una distribución de Poisson es:

$$
P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}
$$

donde $\lambda$ es el parámetro de la distribución, que representa la tasa de ocurrencia del evento.

### 1.29 Función de Distribución Acumulada (CDF)
La CDF de una distribución de Poisson es:

$$
F(k; \lambda) = P(X \leq k) = \sum_{i=0}^{k} \frac{\lambda^i e^{-\lambda}}{i!}
$$

### 1.30 Función de Supervivencia (SF)
La función de supervivencia para una distribución de Poisson es:

$$
S(k; \lambda) = 1 - F(k; \lambda) = 1 - \sum_{i=0}^{k} \frac{\lambda^i e^{-\lambda}}{i!}
$$

### 1.31 Función de Punto Porcentual (PPF)
La función de punto porcentual no tiene una forma cerrada simple para la distribución de Poisson, pero se puede aproximar numéricamente a partir de la CDF inversa.

$$
\text{PPF}(p; \lambda) = \min\{ k \mid F(k; \lambda) \geq p \}
$$

**Recomendación**:

* Si prioriza la concisión, las estimaciones estadísticas y, en general, gráficos atractivos con opciones de personalización razonables, Seaborn es una buena opción para este tipo de gráficos. Aún puede usar las funciones de Matplotlib directamente para personalizaciones específicas dentro de un gráfico de Seaborn si es necesario.

* Si necesita un control absoluto sobre cada elemento de la trama y tiene necesidades de estilo muy específicas que Seaborn no proporciona fácilmente, entonces Matplotlib podría ser una mejor opción, pero podría requerir más codificación.

### 1.32 Uso de Métodos de Clase en las Distribuciones Aleatorias de SciPy

En los ejemplos anteriores, hemos creado instancias de una clase de variables aleatorias y calculado estadísticas y otras propiedades utilizando llamadas a métodos. Sin embargo, una alternativa para utilizar las clases de variables aleatorias en el módulo `stats` de SciPy es usar los métodos de clase directamente.

Por ejemplo, podemos calcular la media de una distribución normal utilizando el método de clase `mean` de `stats.norm`, y pasar los parámetros de la distribución como argumentos (usualmente `loc` y `scale`, como en este caso para valores distribuidos normalmente):

In [ ]:
stats.norm.stats(loc=2, scale=0.5)


lo que da el mismo resultado que crear primero una instancia y luego llamar al
método correspondiente:

In [ ]:
stats.norm(loc=1, scale=0.5).stats()


La mayoría de los métodos en las clases `rv_discrete` y `rv_continuous` pueden ser utilizados como métodos de clase de esta manera.

Hasta ahora, hemos visto solo propiedades de la función de distribución de variables aleatorias. Es importante notar que, aunque una función de distribución describe una variable aleatoria, la distribución en sí misma es completamente determinista. Para generar números aleatorios que estén distribuidos según una distribución de probabilidad dada, podemos usar el método `rvs` (muestra de variable aleatoria). Este método toma como argumento la forma del arreglo requerido (puede ser un número entero para un vector o una tupla con las longitudes de las dimensiones para un arreglo de más de una dimensión).

A continuación, utilizamos `rvs(10)` para generar un arreglo unidimensional con diez valores:

In [ ]:
X = stats.norm(1, 0.5)


In [ ]:
X.rvs(10)


Para ver que los números aleatorios generados están efectivamente distribuidos según la función de distribución de probabilidad correspondiente, podemos graficar un histograma de una gran cantidad de muestras de una variable aleatoria y compararlo con la función de distribución de probabilidad.

Una vez más, para poder hacer esto fácilmente para muestras de varias variables aleatorias, creamos una función `plot_dist_samples` con este propósito. Esta función utiliza el método de intervalo para obtener un rango adecuado de la gráfica para un objeto de variable aleatoria dado.

In [ ]:
def plot_dist_samples(X, X_samples, title=None, ax=None):
    """
    Grafica la PDF y el histograma de muestras de una variable aleatoria continua.
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))

## Obtener el rango adecuado para la gráfica utilizando el método de intervalo
    x_lim = X.interval(.99)
    x = np.linspace(*x_lim, num=100)

## Graficar la PDF de la distribución
    ax.plot(x, X.pdf(x), label="PDF", lw=3)

## Graficar el histograma de las muestras
    ax.hist(X_samples, label="samples", density=1, bins=75)

## Establecer el rango del eje x
    ax.set_xlim(*x_lim)

## Agregar la leyenda
    ax.legend()

## Establecer el título de la gráfica si se proporciona
    if title:
        ax.set_title(title)

    return ax


Cabe destacar que en esta función hemos utilizado la sintaxis de desempaquetado de tuplas `*x_lim`, que distribuye los elementos de la tupla `x_lim` en los diferentes argumentos de la función. En este caso, es equivalente a `np.linspace(x_lim[0], x_lim[1], num=100)`.

A continuación, usamos esta función para visualizar 2000 muestras de tres variables aleatorias con diferentes distribuciones: aquí utilizamos la distribución t de Student, la distribución $\chi^2$ y la distribución exponencial, y los resultados se muestran en la Figura 13-3. Dado que 2000 es una muestra bastante grande, los gráficos del histograma de las muestras coinciden bien con la función de distribución de probabilidad. Con un número aún mayor de muestras, se puede esperar que la concordancia sea aún mejor.

In [ ]:
## Crear una figura con 3 subgráficas
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

## Número de muestras
N = 2000

## Distribución t de Student
X = stats.t(7.0)
plot_dist_samples(X, X.rvs(N), "Distribución t de Student", ax=axes[0])

## Distribución chi-cuadrada
X = stats.chi2(5.0)
plot_dist_samples(X, X.rvs(N), r"$\chi^2$ dist.", ax=axes[1])

## Distribución exponencial
X = stats.expon(0.5)
plot_dist_samples(X, X.rvs(N), "Distribución exponencial", ax=axes[2])

plt.show()


### 1.33 Ajuste de Maxima Verosimilitud: Analisis Completo

El **ajuste de maxima verosimilitud (MLE)** estima parametros que maximizan la probabilidad de observar los datos.

### 1.34 Visualizaciones:
1. **Histograma vs PDF**: Compara distribucion empirica con ajustada
2. **CDF**: Funcion acumulada empirica vs teorica
3. **Q-Q Plot**: Puntos cerca de diagonal indican buen ajuste
4. **Residuos**: Identifica sesgos sistematicos

### 1.35 Tests de Bondad:
- **Kolmogorov-Smirnov**: p-valor > 0.05 indica buen ajuste
- **Anderson-Darling**: Mas sensible en las colas de la distribucion

In [ ]:
## Ajuste de Maxima Verosimilitud - Analisis Completo
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

print('='*70)
print('AJUSTE DE MAXIMA VEROSIMILITUD')
print('='*70)
print('\n1. Generando datos...')
datos_reales = stats.chi2.rvs(df=5, size=500)
print(f'   Muestras: {len(datos_reales)}')
print(f'   Media: {datos_reales.mean():.4f}')

print('\n2. Ajustando distribucion...')
parametros_ajustados = stats.chi2.fit(datos_reales)
df_aj, loc_aj, scale_aj = parametros_ajustados
print(f'   df: {df_aj:.4f}, loc: {loc_aj:.4f}, scale: {scale_aj:.4f}')

dist_ajustada = stats.chi2(df=df_aj, loc=loc_aj, scale=scale_aj)
dist_teorica = stats.chi2(df=5)

print('\n3. Generando visualizaciones...')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

## Histograma vs PDF
ax1 = axes[0, 0]
ax1.hist(datos_reales, bins=50, density=True, alpha=0.6, color='skyblue',
         edgecolor='black', label='Datos reales')
x = np.linspace(0, datos_reales.max(), 1000)
ax1.plot(x, dist_ajustada.pdf(x), 'r-', lw=3, label=f'PDF ajustada (df={df_aj:.2f})')
ax1.plot(x, dist_teorica.pdf(x), 'g--', lw=2, label='PDF teorica (df=5)')
ax1.set_xlabel('Valor', fontsize=12)
ax1.set_ylabel('Densidad', fontsize=12)
ax1.set_title('Comparacion: Datos vs Distribuciones', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

## CDF
ax2 = axes[0, 1]
datos_ord = np.sort(datos_reales)
cdf_emp = np.arange(1, len(datos_ord)+1) / len(datos_ord)
ax2.plot(datos_ord, cdf_emp, 'b-', alpha=0.7, label='CDF empirica', lw=2.5)
ax2.plot(x, dist_ajustada.cdf(x), 'r-', lw=2.5, label='CDF ajustada')
ax2.plot(x, dist_teorica.cdf(x), 'g--', lw=2, label='CDF teorica')
ax2.set_xlabel('Valor', fontsize=12)
ax2.set_ylabel('Probabilidad acumulada', fontsize=12)
ax2.set_title('Funciones de Distribucion Acumulada', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

## Q-Q Plot
ax3 = axes[1, 0]
stats.probplot(datos_reales, dist=stats.chi2, sparams=parametros_ajustados, plot=ax3)
ax3.get_lines()[0].set_marker('o')
ax3.get_lines()[0].set_markersize(4)
ax3.get_lines()[1].set_color('red')
ax3.get_lines()[1].set_linewidth(2)
ax3.set_title('Q-Q Plot: Validacion del Ajuste', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

## Residuos
ax4 = axes[1, 1]
valores_esp = dist_ajustada.ppf(cdf_emp)
residuos = datos_ord - valores_esp
ax4.scatter(valores_esp, residuos, alpha=0.6, s=30, c=residuos, cmap='coolwarm')
ax4.axhline(y=0, color='red', linestyle='--', lw=2, label='Residuo = 0')
ax4.set_xlabel('Valores esperados', fontsize=12)
ax4.set_ylabel('Residuos', fontsize=12)
ax4.set_title('Analisis de Residuos', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n4. Pruebas de bondad de ajuste...')
print('='*70)
ks_stat, ks_pval = stats.kstest(datos_reales, lambda x: dist_ajustada.cdf(x))
print(f'\nTest Kolmogorov-Smirnov:')
print(f'  Estadistico: {ks_stat:.6f}')
print(f'  p-valor: {ks_pval:.6f}')
if ks_pval > 0.05:
    print(f'  RESULTADO: Buen ajuste (no rechazamos H0 al 95%)')
else:
    print(f'  RESULTADO: Ajuste cuestionable')

print('\n' + '='*70)
print('Analisis completado')
print('='*70)


### 1.36 Usos de la Distribución Ajustada (Y)

El objeto $Y$ es una herramienta poderosa para diversas tareas estadísticas. A continuación se detallan algunos de los usos clave:

### 1.37 Cálculo de Probabilidades
Puedes usar Y para calcular probabilidades asociadas con la distribución Chi-cuadrada ajustada. Por ejemplo:

- **Y.pdf(x)**: Calcula la función de densidad de probabilidad (PDF) en un valor dado $x$.
- **Y.cdf(x)**: Calcula la función de distribución acumulada (CDF) en un valor dado $x$, representando la probabilidad de que una variable aleatoria sea menor o igual a $x$.
- **Y.sf(x)**: Calcula la función de supervivencia (SF) en un valor dado $x$, representando la probabilidad de que una variable aleatoria sea mayor que $x$.

### 1.38 Generación de Nuevas Muestras Aleatorias
Puedes generar nuevas muestras aleatorias que sigan la distribución Chi-cuadrada ajustada utilizando **Y.rvs(size)**. Esto puede ser útil para simulaciones o análisis posteriores.

### 1.39 Pruebas de Hipótesis
La distribución Chi-cuadrada se utiliza frecuentemente en pruebas de hipótesis, especialmente para pruebas de bondad de ajuste y pruebas de independencia. Puedes usar $Y$ para calcular valores $p$ y tomar decisiones sobre hipótesis.

### 1.40 Intervalos de Confianza
Puedes usar Y para calcular intervalos de confianza para los parámetros relacionados con la distribución Chi-cuadrada.

### 1.41 Modelado Estadístico
La distribución ajustada Y puede servir como un modelo estadístico para los datos que estás analizando. Este modelo puede usarse para hacer predicciones o entender patrones subyacentes.

### 1.42 Ejemplos de Escenarios

Aquí tienes algunos ejemplos para ilustrar cómo podrías usar la distribución Chi-cuadrada ajustada en la práctica:

### 1.43 Prueba de Bondad de Ajuste
Si tienes datos observados y deseas probar si siguen una distribución Chi-cuadrada, podrías usar el método `fit` para estimar los parámetros y luego comparar los datos observados con la distribución ajustada mediante una prueba de bondad de ajuste.

### 1.44 Predicción de Valores Futuros
Si tienes un proceso que genera datos que parecen seguir una distribución Chi-cuadrada, podrías usar la distribución ajustada Y para predecir la probabilidad de observar ciertos valores en el futuro.

### 1.45 Simulación de Experimentos
Podrías usar **Y.rvs()** para generar datos aleatorios que sigan la distribución ajustada para simulaciones o análisis de Monte Carlo.

### 1.46 En Resumen

La distribución Chi-cuadrada ajustada Y proporciona una forma de modelar y entender tus datos al ajustarlos a una distribución de probabilidad conocida. Este modelo ajustado se puede usar para diversas tareas estadísticas, como calcular probabilidades, hacer predicciones y probar hipótesis. Es una herramienta fundamental para el análisis estadístico y la ciencia de datos. ¡Espero que esto ayude a clarificar su propósito y cómo puedes usarlo de manera efectiva!

In [ ]:
x=5
## Y=parametros_ajustados # This was a tuple of parameters, not a distribution object
## Create the chi-squared distribution object using the fitted parameters
df_aj, loc_aj, scale_aj = parametros_ajustados
Y = stats.chi2(df=df_aj, loc=loc_aj, scale=scale_aj)
Y.pdf(x)


In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np

def plot_rv_distribution(rv):
    """
    Plots the PDF, CDF, SF, and PPF of a given random variable.

    Args:
        rv: A SciPy stats random variable object.
    """
## Create x-axis values for plotting
    x = np.linspace(rv.ppf(0.01), rv.ppf(0.99), 100)

## Create subplots for PDF, CDF, SF, and PPF
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

## Plot PDF
    axes[0, 0].plot(x, rv.pdf(x), 'r-', lw=2, label='PDF')
    axes[0, 0].set_title('Probability Density Function (PDF)')

## Plot CDF
    axes[0, 1].plot(x, rv.cdf(x), 'b-', lw=2, label='CDF')
    axes[0, 1].set_title('Cumulative Distribution Function (CDF)')

## Plot SF (Survival Function)
    axes[1, 0].plot(x, rv.sf(x), 'g-', lw=2, label='SF')
    axes[1, 0].set_title('Survival Function (SF)')

## Plot PPF (Percent Point Function)
    axes[1, 1].plot(x, rv.ppf(x), 'm-', lw=2, label='PPF')
    axes[1, 1].set_title('Percent Point Function (PPF)')

## Add legends and grid to subplots
    for ax in axes.flatten():
        ax.legend(loc='best', frameon=False)
        ax.grid(True)

## Adjust spacing between subplots
    plt.tight_layout()

## Define Y as a normal random variable with mean 1 and standard deviation 0.5
Y = stats.norm(1, 0.5)

## Generate the plots (PDF, CDF, SF, PPF)
plot_rv_distribution(Y)

## Show the plot
plt.show()


### 1.47 Otros Métodos para determinar la distribución de una muestra

Existen métodos para intentar determinar la distribución de una muestra sin necesidad de identificar visualmente una posible distribución de antemano. Estos métodos se basan en algoritmos que exploran un conjunto de distribuciones candidatas y seleccionan la que mejor se ajusta a los datos según ciertos criterios.

Aquí dos enfoques:

### 1.48 **Algoritmos de selección de distribuciones**:

Existen bibliotecas en Python que implementan algoritmos para la selección automática de distribuciones, como `distfit` y `best_fit`.
Estas bibliotecas suelen utilizar una combinación de pruebas de bondad de ajuste, criterios de información (**AIC, BIC**) y otras métricas para evaluar el ajuste de diferentes distribuciones a los datos.
El usuario proporciona la muestra de datos y el algoritmo devuelve la distribución que mejor se ajusta según los criterios especificados.

In [ ]:
!pip install distfit

from distfit import distfit

## Generar una muestra de datos
data = np.random.normal(loc=0, scale=1, size=1000)

## Instanciar el objeto distfit
dist = distfit()

## Ajustar las distribuciones a los datos
dist.fit_transform(data)

## Mostrar la mejor distribución encontrada
print(dist.model)

## Visualizar el ajuste
dist.plot()


### 1.49 Red Neuronal Probabilistica: Analisis Completo

Las **redes neuronales probabilisticas** combinan deep learning con teoria de probabilidad.

### 1.50 Arquitectura:
- 3 capas ocultas (64, 32, 16) con ReLU y Dropout
- Capa de salida: `DistributionLambda` (genera distribucion Normal)
- Loss: Negative Log-Likelihood

### 1.51 Evaluacion:
- **Train/Test split**: 80/20
- **Metricas**: MAE, RMSE, R2 para ambos conjuntos
- **Visualizaciones**: 9 graficas incluyendo curvas de aprendizaje, predicciones, residuos

### 1.52 Interpretacion:
- **Delta (train-test) pequeno**: Modelo generaliza bien
- **Delta grande**: Posible sobreajuste
- **R2 cercano a 1**: Excelente ajuste

### 1.53 Comparacion con Metodos Clasicos:

| Aspecto | MLE Clasico | Red Neuronal |
|---------|-------------|---------------|
| Velocidad | Rapido | Lento |
| Interpretabilidad | Alta | Baja |
| Flexibilidad | Baja | Alta |
| Datos requeridos | Pocos | Muchos |

In [ ]:
## RED NEURONAL PROBABILISTICA: Analisis Train/Test Completo
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from scipy import stats

tfd = tfp.distributions
tfpl = tfp.layers

np.random.seed(42)
tf.random.set_seed(42)

print('='*80)
print(' '*20 + 'RED NEURONAL PROBABILISTICA')
print('='*80)

## 1. DATOS
print('\n[1/6] Generando datos...')
n_samples = 2000
X_data = np.random.randn(n_samples, 1).astype(np.float32)
y_data = X_data.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42)

print(f'   Total: {n_samples} muestras')
print(f'   Train: {X_train.shape[0]} ({X_train.shape[0]/n_samples*100:.0f}%)')
print(f'   Test: {X_test.shape[0]} ({X_test.shape[0]/n_samples*100:.0f}%)')

## 2. MODELO
print('\n[2/6] Construyendo modelo...')
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(2),  # Output layer now directly outputs loc and scale
])

## Modified loss function to create distribution from model output
def neg_log_likelihood(y_true, y_pred_params):
## y_pred_params will be the output of the last Dense layer (loc and scale)
    loc, scale = tf.split(y_pred_params, num_or_size_splits=2, axis=-1)
## Ensure scale is positive using softplus
    scale = tf.math.softplus(scale)
## Create the distribution
    y_pred_dist = tfd.Normal(loc=loc, scale=scale)
## Compute the negative log probability
    return -y_pred_dist.log_prob(y_true)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=neg_log_likelihood,
## Can add metrics that work with tensor output, e.g., 'mae' if desired,
## but be mindful that this is comparing the mean of the predicted distribution
## to the true value, not evaluating the full distribution fit.
## For now, keeping metrics simple or removed to avoid potential issues.
## metrics=['mae'] # Re-add MAE if needed and compatible
)

print(f'   Parametros: {model.count_params():,}')

## 3. ENTRENAR
print('\n[3/6] Entrenando...')
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_split=0.2, epochs=200, batch_size=32, callbacks=[early_stop], verbose=0)

print(f'   Epocas: {len(history.history["loss"])}')

## 4. PREDICCIONES
print('\n[4/6] Predicciones...')
## When predicting after training, get the parameters and create the distribution
y_pred_params_train = model(X_train)
y_pred_dist_train = tfd.Normal(loc=y_pred_params_train[..., :1], scale=tf.math.softplus(y_pred_params_train[..., 1:]))
y_pred_train = y_pred_dist_train.mean().numpy()

y_pred_params_test = model(X_test)
y_pred_dist_test = tfd.Normal(loc=y_pred_params_test[..., :1], scale=tf.math.softplus(y_pred_params_test[..., 1:]))
y_pred_test = y_pred_dist_test.mean().numpy()

## 5. METRICAS
print('\n[5/6] Calculando metricas...')
mae_train = mean_absolute_error(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r2_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

print('\n' + '='*80)
print(' '*25 + 'METRICAS DE RENDIMIENTO')
print('='*80)
print(f'\nTRAIN:')
print(f'  MAE:  {mae_train:.6f}')
print(f'  RMSE: {rmse_train:.6f}')
print(f'  R2:   {r2_train:.6f}')

print(f'\nTEST:')
print(f'  MAE:  {mae_test:.6f}')
print(f'  RMSE: {rmse_test:.6f}')
print(f'  R2:   {r2_test:.6f}')

print(f'\nDIFERENCIA (sobreajuste):')
print(f'  Delta MAE:  {abs(mae_train-mae_test):.6f}')
print(f'  Delta RMSE: {abs(rmse_train-rmse_test):.6f}')
print(f'  Delta R2:   {abs(r2_train-r2_test):.6f}')
print('='*80)

## 6. VISUALIZACIONES
print('\n[6/6] Generando visualizaciones...')

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

## Loss
ax1 = fig.add_subplot(gs[0, 0])
epochs = range(1, len(history.history['loss'])+1)
ax1.plot(epochs, history.history['loss'], 'b-', lw=2, label='Train', alpha=0.8)
ax1.plot(epochs, history.history['val_loss'], 'r-', lw=2, label='Val', alpha=0.8)
ax1.set_xlabel('Epoca', fontsize=11, fontweight='bold')
ax1.set_ylabel('Loss', fontsize=11, fontweight='bold')
ax1.set_title('Curva de Aprendizaje: Loss', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

## MAE
## MAE is calculated manually after training, not available in history
## ax2 = fig.add_subplot(gs[0, 1])
## ax2.set_xlabel('Epoca', fontsize=11, fontweight='bold')
## ax2.set_ylabel('MAE', fontsize=11, fontweight='bold')
## ax2.set_title('Curva de Aprendizaje: MAE (Calculated Post-Training)', fontsize=13, fontweight='bold')
## ax2.grid(True, alpha=0.3)
## Placeholder for MAE plot if history was available or calculated per epoch

## Barras metricas
ax3 = fig.add_subplot(gs[0, 2])
metricas = ['MAE', 'RMSE', 'R2']
train_vals = [mae_train, rmse_train, r2_train]
test_vals = [mae_test, rmse_test, r2_test]
x = np.arange(len(metricas))
w = 0.35
bars1 = ax3.bar(x-w/2, train_vals, w, label='Train', alpha=0.8, color='steelblue')
bars2 = ax3.bar(x+w/2, test_vals, w, label='Test', alpha=0.8, color='coral')
ax3.set_xticks(x)
ax3.set_xticklabels(metricas)
ax3.set_ylabel('Valor', fontsize=11, fontweight='bold')
ax3.set_title('Metricas: Train vs Test', fontsize=13, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax3.text(bar.get_x()+bar.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=8)

## Scatter Train
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(y_train, y_pred_train, alpha=0.4, s=25, c='steelblue')
lims = [min(y_train.min(), y_pred_train.min()), max(y_train.max(), y_pred_train.max())]
ax4.plot(lims, lims, 'r--', lw=2.5, label='Ideal', zorder=5)
ax4.set_xlabel('Real', fontsize=11, fontweight='bold')
ax4.set_ylabel('Predicho', fontsize=11, fontweight='bold')
ax4.set_title(f'Train: Real vs Pred (R2={r2_train:.4f})', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_aspect('equal', adjustable='box')

## Scatter Test
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(y_test, y_pred_test, alpha=0.4, s=25, c='coral')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax5.plot(lims, lims, 'r--', lw=2.5, label='Ideal', zorder=5)
ax5.set_xlabel('Real', fontsize=11, fontweight='bold')
ax5.set_ylabel('Predicho', fontsize=11, fontweight='bold')
ax5.set_title(f'Test: Real vs Pred (R2={r2_test:.4f})', fontsize=13, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)
ax5.set_aspect('equal', adjustable='box')

## Combinado
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(y_train, y_pred_train, alpha=0.3, s=20, c='steelblue', label='Train')
ax6.scatter(y_test, y_pred_test, alpha=0.3, s=20, c='coral', label='Test')
lims = [
    min(y_train.min(), y_test.min(), y_pred_train.min(), y_pred_test.min()),
    max(y_train.max(), y_test.max(), y_pred_train.max(), y_pred_test.max())
]
ax6.plot(lims, lims, 'r--', lw=2.5, label='Ideal', zorder=5)
ax6.set_xlabel('Real', fontsize=11, fontweight='bold')
ax6.set_ylabel('Predicho', fontsize=11, fontweight='bold')
ax6.set_title('Comparacion Train/Test', fontsize=13, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.set_aspect('equal', adjustable='box')

## Residuos Train
ax7 = fig.add_subplot(gs[2, 0])
res_train = (y_train - y_pred_train).flatten()
ax7.scatter(y_pred_train, res_train, alpha=0.4, s=25, c='steelblue')
ax7.axhline(y=0, color='red', linestyle='--', lw=2.5, label='Res=0')
ax7.set_xlabel('Predicho', fontsize=11, fontweight='bold')
ax7.set_ylabel('Residuo', fontsize=11, fontweight='bold')
ax7.set_title('Residuos: Train', fontsize=13, fontweight='bold')
ax7.legend()
ax7.grid(True, alpha=0.3)

## Residuos Test
ax8 = fig.add_subplot(gs[2, 1])
res_test = (y_test - y_pred_test).flatten()
ax8.scatter(y_pred_test, res_test, alpha=0.4, s=25, c='coral')
ax8.axhline(y=0, color='red', linestyle='--', lw=2.5, label='Res=0')
ax8.set_xlabel('Predicho', fontsize=11, fontweight='bold')
ax8.set_ylabel('Residuo', fontsize=11, fontweight='bold')
ax8.set_title('Residuos: Test', fontsize=13, fontweight='bold')
ax8.legend()
ax8.grid(True, alpha=0.3)

## Distribucion errores
ax9 = fig.add_subplot(gs[2, 2])
ax9.hist(res_train, bins=50, alpha=0.6, label='Train', color='steelblue', density=True, edgecolor='black')
ax9.hist(res_test, bins=50, alpha=0.6, label='Test', color='coral', density=True, edgecolor='black')
x_norm = np.linspace(res_train.min(), res_train.max(), 100)
ax9.plot(x_norm, stats.norm.pdf(x_norm, 0, res_train.std()), 'g--', lw=2, label='Normal')
ax9.set_xlabel('Error', fontsize=11, fontweight='bold')
ax9.set_ylabel('Densidad', fontsize=11, fontweight='bold')
ax9.set_title('Distribucion de Errores', fontsize=13, fontweight='bold')
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.suptitle('RED NEURONAL: Analisis Completo Train/Test', fontsize=16, fontweight='bold', y=0.997)
plt.show()

print('\n' + '='*80)
print('Analisis completo finalizado')
print('='*80)


### 1.54 Guardar el Modelo Entrenado

Puedes guardar tu modelo usando el método `model.save()`. Especifica la ruta donde quieres guardar el modelo.

In [ ]:
## Define la ruta donde quieres guardar el modelo
model_save_path = 'probabilistic_model.keras' # Added .keras extension

## Guarda el modelo en formato SavedModel
model.save(model_save_path)

print(f"Modelo guardado en: {model_save_path}")


### 1.55 Cargar el Modelo Guardado

Para usar el modelo más tarde, puedes cargarlo usando `tf.keras.models.load_model()`. Es importante que la función de pérdida personalizada (`neg_log_likelihood`) esté disponible en el entorno donde cargas el modelo, ya que es necesaria para la configuración del modelo.

In [ ]:
## Carga el modelo guardado
## Asegúrate de que la funcion neg_log_likelihood este definida en este entorno
loaded_model = tf.keras.models.load_model(model_save_path, custom_objects={'neg_log_likelihood': neg_log_likelihood})

print("Modelo cargado exitosamente.")


### 1.56 Usar el Modelo Cargado para Cálculos Probabilísticos

Una vez que el modelo está cargado, puedes usarlo para hacer predicciones en nuevos datos. La salida del modelo cargado seguirá siendo un objeto de distribución de TensorFlow Probability, lo que te permite realizar cálculos probabilísticos como obtener la media, la desviación estándar, o calcular la probabilidad de observar ciertos valores.

### 1.57 Problema contextualizado en Nanotecnología: Control de Calidad en Nanopartículas

Imagina que estás trabajando en un laboratorio de nanotecnología sintetizando nanopartículas de oro para aplicaciones biomédicas. El tamaño de las nanopartículas es crucial para su eficacia y seguridad. Durante el proceso de síntesis, controlas un parámetro clave, como la concentración de un agente reductor. Sabes que este parámetro influye en el tamaño final de las nanopartículas, pero el proceso tiene una variabilidad inherente.

Has recopilado datos experimentales donde registraste la concentración del agente reductor (`X`) y el tamaño promedio de las nanopartículas obtenidas (`y`). Entrenaste el modelo de red neuronal probabilística que acabamos de desarrollar para predecir el tamaño de la nanopartícula (`y`) basándose en la concentración del agente reductor (`X`).

Ahora, quieres utilizar tu modelo para:

1.  **Predecir el tamaño esperado** de las nanopartículas para nuevas concentraciones del agente reductor.
2.  **Cuantificar la incertidumbre** en esas predicciones (es decir, saber cuán variable podría ser el tamaño real alrededor de la predicción promedio).
3.  **Calcular la probabilidad** de que el tamaño de las nanopartículas caiga dentro de un rango específico o exceda un cierto umbral para una concentración dada. Esto es vital para el control de calidad, ya que solo se aceptan lotes de nanopartículas dentro de un rango de tamaño muy estrecho.

**Ejemplo de uso:**

Supongamos que quieres predecir el tamaño de las nanopartículas para concentraciones del agente reductor de 1.5, -0.8, 0.0 y 2.1 (estos valores, aunque simplificados para el ejemplo, representarían tus niveles controlados del parámetro X).

Utilizarías el código que ya tenemos para:

*   Obtener la media predicha del tamaño para cada concentración.
*   Obtener la desviación estándar predicha del tamaño para cada concentración (esta es tu medida de incertidumbre).
*   Calcular, por ejemplo, la probabilidad de que el tamaño de las nanopartículas sea mayor a 1.8 nm (un umbral de control de calidad) cuando la concentración del agente reductor es 1.5.

Este problema ilustra cómo el modelo probabilístico va más allá de una simple predicción puntual, proporcionando información valiosa sobre la variabilidad y permitiendo cálculos de probabilidad esenciales para el control de calidad en un contexto de nanotecnología.

In [ ]:
## Genera algunos datos nuevos para prediccion
new_data = np.array([[1.5], [-0.8], [0.0], [2.1]], dtype=np.float32)

## Realiza predicciones con el modelo cargado
## The loaded model outputs the raw parameters (loc and scale)
predicted_params = loaded_model(new_data)

## Explicitly create the distribution object from the predicted parameters
loc, scale = tf.split(predicted_params, num_or_size_splits=2, axis=-1)
## Ensure scale is positive using softplus, consistent with training
scale = tf.math.softplus(scale)
predicted_distribution = tfd.Normal(loc=loc, scale=scale)

## Ahora puedes usar los metodos de la distribucion para calculos probabilisticos
mean_predictions = predicted_distribution.mean().numpy()
stddev_predictions = predicted_distribution.stddev().numpy()
## You can also sample from the distribution
## samples = predicted_distribution.sample(100)

print("Predicciones (Media):")
print(mean_predictions)

print("\nPredicciones (Desviacion Estandar):")
print(stddev_predictions)

## Ejemplo: Calcular la probabilidad de que el primer nuevo dato sea mayor que 1.0
## This is equivalent to 1 - CDF(1.0)
probability_greater_than_1 = 1.0 - predicted_distribution.cdf(1.0).numpy()
print(f"\nProbabilidad de que el primer dato sea > 1.0: {probability_greater_than_1[0][0]:.4f}")


### 1.58 Interpretación de los Resultados en el Contexto del Problema de Nanotecnología

Recordemos que estás prediciendo el **tamaño promedio de las nanopartículas** (`y`) basado en la **concentración del agente reductor** (`X`). Las `Predicciones (Media)` te dan el tamaño promedio esperado, y las `Predicciones (Desviacion Estandar)` te dan la incertidumbre o variabilidad esperada alrededor de ese tamaño promedio para cada concentración de agente reductor que probaste (`new_data`).

Interpretación de los resultados:

1.  **Predicciones (Media):**
    *   Para la concentración de agente reductor de **1.5**, el modelo predice un tamaño promedio de nanopartícula de aproximadamente **1.397 nm**.
    *   Para la concentración de **-0.8**, el tamaño promedio predicho es de aproximadamente **-0.752 nm**. (Nota: Un tamaño negativo no tiene sentido físico. Esto podría indicar que la concentración de -0.8 está fuera del rango de los datos de entrenamiento donde el modelo se comporta bien, o que se necesita una transformación en los datos de salida para asegurar predicciones positivas).
    *   Para la concentración de **0.0**, el tamaño promedio predicho es de aproximadamente **-0.012 nm**. (Similar al punto anterior, sugiere revisar el rango de datos o aplicar transformaciones).
    *   Para la concentración de **2.1**, el tamaño promedio predicho es de aproximadamente **1.900 nm**.

2.  **Predicciones (Desviación Estándar):**
    *   Para la concentración de **1.5**, la desviación estándar predicha es de aproximadamente **0.193 nm**. Esto significa que, aunque el tamaño promedio esperado es 1.397 nm, el tamaño real de las nanopartículas en un lote sintetizado con esta concentración probablemente variará alrededor de este valor con una dispersión típica de 0.193 nm.
    *   Para la concentración de **-0.8**, la desviación estándar es de aproximadamente **0.088 nm**. La variabilidad predicha es menor aquí.
    *   Para la concentración de **0.0**, la desviación estándar es de aproximadamente **0.036 nm**. La variabilidad predicha es aún menor.
    *   Para la concentración de **2.1**, la desviación estándar es de aproximadamente **0.208 nm**. La variabilidad predicha es la más alta entre estos ejemplos.

    La desviación estándar es tu medida de **incertidumbre** o la **variabilidad esperada en el tamaño de las nanopartículas**. Un valor más alto indica que es probable que los tamaños de las nanopartículas dentro de un lote estén más dispersos alrededor del tamaño promedio predicho. Esto es crucial para el control de calidad, ya que te dice cuán consistente es probable que sea el proceso de síntesis para una concentración dada del agente reductor.

3.  **Probabilidad de que el primer dato sea > 1.0: 0.9805**
    *   Esta línea se refiere a la primera entrada en `new_data`, que corresponde a una concentración de agente reductor de **1.5**.
    *   El resultado de **0.9805** significa que, basado en tu modelo probabilístico, hay una probabilidad del **98.05%** de que el tamaño de las nanopartículas sintetizadas con una concentración de agente reductor de 1.5 nm sea **mayor que 1.0 nm**.

**Implicaciones para el Control de Calidad:**

*   Si tu criterio de control de calidad requiere que el tamaño de las nanopartículas esté en un rango específico (por ejemplo, entre 1.2 nm y 1.6 nm), no solo mirarías la media predicha (1.397 nm para X=1.5), sino también la desviación estándar (0.193 nm). Una desviación estándar alta podría significar que, aunque la media esté en el rango, una porción significativa de las nanopartículas podría estar fuera del rango aceptable debido a la variabilidad del proceso.
*   La probabilidad calculada (98.05% de que el tamaño sea > 1.0 nm para X=1.5) es un ejemplo directo de cómo puedes usar el modelo para tomar decisiones de control de calidad. Si, por ejemplo, la especificación mínima de tamaño es 1.0 nm, una probabilidad del 98.05% de estar por encima de ese umbral es una información valiosa para decidir si un lote producido bajo estas condiciones es aceptable. Podrías calcular la probabilidad de estar *dentro* de tu rango de tamaño objetivo de manera similar.
*   Los resultados para concentraciones de -0.8 y 0.0 sugieren que estas concentraciones, si bien predicen desviaciones estándar bajas, dan como resultado tamaños promedio que no son físicamente posibles. Esto subraya la importancia de entender el dominio de tus datos y el comportamiento del modelo fuera del rango de entrenamiento.

Este análisis te permite ir más allá de simplemente predecir un valor puntual y te da una comprensión de la incertidumbre asociada, lo cual es fundamental para optimizar tus procesos de síntesis y establecer criterios de control de calidad basados en probabilidades.

#Investigación

1. ¿Qué es?

La Prueba de Hipótesis Estadística es un método para abordar el problema de realizar una afirmación sobre un parámetro desconocido asociado a una distribución de probabilidad, basándose en la información de una muestra aleatoria.En lugar de encontrar una estimación para el parámetro, se propone un valor (o un rango de valores) para el parámetro desconocido (hipótesis) y luego se utiliza la información de la muestra para confirmar o refutar este valor hipotetizado. La hipótesis estadística paramétrica es la verificación de una aseveración sobre el parámetro desconocido ($\theta$) con la ayuda de observaciones obtenidas de una muestra aleatoria.

* ¿Cómo se define?

El proceso se define a través de dos afirmaciones complementarias y el uso de regiones de decisión:

Hipótesis Nula ($H_0$): Es la aseveración sobre el parámetro desconocido ($\theta$), denotada formalmente como $H_0: \theta \in \Theta_0$. El signo de "igual a" ($=$) siempre se incluye en la hipótesis nula.

Hipótesis Alternativa ($H_1$): Es la afirmación complementaria a la hipótesis nula, denotada como $H_1: \theta \in \Theta_1 = \Theta \setminus \Theta_0$.

Decisión: Se utiliza un Estadístico de Prueba y una Región Crítica ($\Omega$). Si el valor de la muestra ($x$) cae en la región crítica ($x \in \Omega$), se rechaza la hipótesis nula ($H_0$) a favor de la alternativa ($H_1$); si $x \notin \Omega$, no se rechaza $H_0$.

Una hipótesis es simple si $\Theta_0$ y $\Theta_1$ consisten en un solo punto; de lo contrario, es compuesta.

* ¿Para qué sirve?

El propósito principal es verificar (contrastar) una aseveración sobre una característica de la población (un parámetro) basándose en evidencia limitada (la muestra).Esto permite a ingenieros y científicos:Tomar decisiones sobre si un proceso o fenómeno se ajusta a un modelo teórico con parámetros específicos.Cuantificar la incertidumbre de una conclusión al definir la probabilidad de cometer un error de tipo I ($\alpha$, nivel de significancia) o un error de tipo II ($\beta$).

* ¿Dónde se utiliza?

El método se utiliza en una amplia gama de contextos científicos e ingenieriles:
Control de Calidad: Probar si la media de un producto (por ejemplo, el rendimiento de un portafolio de acciones o la duración de un lote de neumáticos) es igual a un valor objetivo.

Ciencia de Datos y Modelado: Verificar si una distribución de probabilidad (e.g., normal, exponencial) es adecuada para modelar un conjunto de datos.

Finanzas y Economía: Probar si el rendimiento promedio de un activo (e.g., el petróleo crudo Brent Blend) es significativamente diferente de cero.

Investigación Social y Correlación: Determinar si dos poblaciones o variables (e.g., velocidad del tráfico y volumen del tráfico) están correlacionadas o son independientes

* ¿Cómo se calcula o aplica?

El procedimiento general, ilustrado con la prueba de media poblacional ($\mu$) con varianza ($\sigma^2$) conocida (Z-test), es el siguiente:

Planteamiento de Hipótesis: Definir $H_0: \mu = \mu_0$ y $H_1: \mu \ne \mu_0$.

Estadístico de Prueba: Se utiliza el estadístico $Z$:$Z = \frac{\overline{X} - \mu_0}{\sigma/\sqrt{n}}$Bajo $H_0$, $Z \sim N(0, 1)$.

Cálculo del Valor Observado ($z_0$): Sustituir el valor muestral ($\overline{x}$) y los parámetros de $H_0$ en la fórmula:$z_0 = \frac{\overline{x} - \mu_0}{\sigma/\sqrt{n}}$

Región Crítica y Nivel de Significancia ($\alpha$): Dado el nivel de significancia $\alpha$, se encuentran los valores críticos ($z_{\alpha/2}$)

Decisión: Se acepta $H_0$ si el valor calculado $z_0$ está en el intervalo de aceptación (es decir, $-z_{\alpha/2} < z_0 < z_{\alpha/2}$). Si $z_0$ cae fuera de este rango, se rechaza $H_0$.

* Qué ejemplos ilustran su aplicación?

Ejemplo (Z-Test para la media)Se prueba la hipótesis $H_0: \mu=8$ contra $H_1: \mu \ne 8$ con un nivel de significancia $\alpha=0.01$, dado $\sigma^2=0.25$ (conocida), media muestral $\overline{x}=7.8$ y tamaño de muestra $n=50$.

Cálculo del Estadístico de Prueba ($z_0$):$z_{0} = \frac{\overline{x} - \mu_{0}}{\sigma/\sqrt{n}} = \frac{7.8 - 8}{\sqrt{0.25}/\sqrt{50}} = \frac{-0.2}{0.5/\sqrt{50}} \approx -2.83$.

Región Crítica:Para $\alpha=0.01$ (prueba de dos colas), los valores críticos son $\pm z_{0.005} = \pm 2.57$. Se acepta $H_0$ si $-2.57 < z_0 < 2.57$.

Conclusión:Como $z_{0} = -2.83$ es menor que $-2.57$, el estadístico cae en la región crítica. Por lo tanto, se rechaza la hipótesis nula ($H_0$).

* ¿Cómo se valida o comprueba?

La validación de la prueba de hipótesis se realiza de dos maneras principales33:Método del Valor Crítico (Región de Rechazo): Se verifica si el valor calculado del estadístico de prueba (e.g., $z_0$, $t_0$, $\chi^2_0$) se encuentra dentro de la región crítica ($C$) definida por el nivel de significancia $\alpha$. Si el estadístico de prueba cae en $C$, se rechaza $H_0$.

Método del p-valor: Se calcula el p-valor, que es la probabilidad de obtener un resultado igual o más extremo que el observado, bajo el supuesto de que $H_0$ es verdadera. La decisión se toma comparando el p-valor con $\alpha$:Rechazar $H_0$ si p-valor $< \alpha$.

*Qué limitaciones o supuestos tiene?

Limitación sobre Errores: Es imposible minimizar simultáneamente la probabilidad de cometer un Error Tipo I ($\alpha$) (rechazar $H_0$ siendo verdadera) y un Error Tipo II ($\beta$) (no rechazar $H_0$ siendo falsa).

Generalmente, se fija $\alpha$ y se busca minimizar $\beta$.Supuestos (Z-test): Para la prueba de la media con varianza conocida, se asume que la muestra proviene de una distribución normal.

Supuestos (T-test): Si la varianza es desconocida, se reemplaza por la varianza muestral ($s^2$), y el estadístico sigue una distribución t con $n-1$ grados de libertad.

Si $n \ge 30$, la distribución $t$ puede ser reemplazada por la normal estándar.

* ¿Qué otras variantes o extensiones existen?

Existen múltiples extensiones y pruebas, entre ellas:

Lema de Neyman-Pearson (NP-Lemma): Un teorema fundamental que establece la prueba más potente para un contraste de dos hipótesis simples, al maximizar la potencia ($1-\beta$) para un nivel de significancia $\alpha$ dado.

Prueba de Razón de Verosimilitud (Likelihood Ratio Test): Una generalización del NP-Lemma utilizada cuando no se puede encontrar la región crítica a través del Lema de NP.

Prueba para la Diferencia entre Dos Medias: Se utiliza para comparar si las medias de dos poblaciones independientes son iguales ($H_0: \mu_1 - \mu_2 = \delta$), usando el estadístico $Z$ o $T$ según se conozcan o no las varianzas.

Prueba para la Varianza ($\sigma^2$): Se utiliza para contrastar hipótesis sobre la varianza de una población normal ($H_0: \sigma^2 = \sigma_0^2$), empleando la distribución Chi-cuadrado ($\chi^2$) como estadístico de prueba.

### 1.59 Sección 5.1: Prueba de Hipótesis Estadística (Testing of Statistical Hypothesis)

Esta sección aborda el proceso de utilizar la información de una muestra aleatoria para confirmar o refutar un valor **hipotetizado** (una aseveración) sobre un parámetro desconocido $\theta$ asociado a una distribución de probabilidad $f(x; \theta)$.

5.1.1 Conceptos Fundamentales, Errores y P-value

Hipótesis Nula ($H_0$) y Alternativa ($H_1$)
Una hipótesis estadística paramétrica es la verificación de una aseveración sobre el parámetro desconocido $\theta$.

*   Hipótesis Nula ($H_0$):Es la aseveración sobre el parámetro $\theta$. Se denota como $H_0: \theta \in \Theta_0$. Una regla clave es que el signo de 'igual a' (=), 'mayor o igual a' ($\ge$), o 'menor o igual a' ($\le$) siempre se incluye en la hipótesis nula.
*   Hipótesis Alternativa ($H_1$):Es la afirmación complementaria a la hipótesis nula. Se denota como $H_1: \theta \in \Theta_1 = \Theta \setminus \Theta_0$.

Estadístico de Prueba (Test Statistic) y Región Crítica
*   Estadístico de Prueba: Es una estadística cuyo valor se determina usando la realización de la muestra.
*   Región Crítica (o Región de Rechazo): Es el conjunto de valores del estadístico de prueba para los cuales la hipótesis nula debe ser rechazada. Si la observación cae en esta región, se rechaza $H_0$ a favor de $H_1$.

Tipos de Errores
Al realizar una prueba de hipótesis, se pueden cometer dos tipos de errores estadísticos:

| | $H_0$ es verdadera | $H_0$ es falsa |
| :---: | :---: | :---: |
| **No rechazar $H_0$** | Inferencia Correcta | **Error Tipo II** ($\beta$) |
| **Rechazar $H_0$** | **Error Tipo I** ($\alpha$) | Inferencia Correcta |

*   Error Tipo I: Rechazar $H_0$ cuando, de hecho, es verdadera.
    *   La probabilidad de cometer este error se llama nivel de significancia ($\alpha$). $\alpha = P(\text{rechazar } H_0 \mid H_0 \text{ es verdadera})$.
*   Error Tipo II: No rechazar $H_0$ cuando, en realidad, es falsa.
    *   La probabilidad de cometer este error se denota por $\beta$. $\beta = P(\text{aceptar } H_0 \mid H_0 \text{ es falsa})$.
*   Poder de la Prueba: Es $1 - \beta$, la probabilidad de rechazar $H_0$ cuando es falsa.

Ejercicio Explicado (Ejemplo 5.1): Cálculo de Errores

Contexto: Sea $X \sim \text{Exp}(\lambda)$. Se prueban $H_0: \mu = 20$ contra $H_1: \mu = 30$, donde $\mu = 1/\lambda$. Se observa una muestra de tamaño uno ($x$), y la regla de prueba es rechazar $H_0$ si $x > 28$.

1. Probabilidad de Error Tipo I ($\alpha$):
$\alpha = P(\text{rechazar } H_0 \mid H_0 \text{ es verdadera})$.
Si $H_0$ es verdadera, $\mu=20$, por lo tanto $\lambda = 1/20$.
$$P(\text{Tipo I}) = P(X > 28 \mid X \sim \text{Exp}(1/20))$$
$$P(\text{Tipo I}) = 1 - F_X(28) = e^{-28/20} = \mathbf{0.2466}$$.

2. Probabilidad de Error Tipo II ($\beta$):
$\beta = P(\text{no rechazar } H_0 \mid H_0 \text{ es falsa})$.
Si $H_0$ es falsa, $H_1$ es verdadera, $\mu=30$, por lo tanto $\lambda = 1/30$. No rechazar $H_0$ significa que $X \le 28$.
$$P(\text{Tipo II}) = P(X \le 28 \mid X \sim \text{Exp}(1/30))$$
$$P(\text{Tipo II}) = F_X(28) = 1 - e^{-28/30} = \mathbf{0.6068}$$.

P-value
El p-value asociado a una prueba es la probabilidad de obtener un resultado igual o más extremo que el observado, bajo la suposición de que $H_0$ es verdadera.

*   Decisión por P-value: Se rechaza la hipótesis nula si el p-value es menor que $\alpha$.

5.1.2 Teoría de Neyman–Pearson (NP-Lemma)

La teoría de Neyman–Pearson busca un "buen test". Para un valor fijo de $\alpha$ (probabilidad de Error Tipo I), se intenta **minimizar la probabilidad de Error Tipo II ($\beta$). Un test que logra esto se llama el test más potente (most powerful test).

Teorema 5.1 (Lema de Neyman–Pearson (NP-Lemma))
Para una prueba simple $H_0: \theta = \theta_0$ contra una alternativa simple $H_1: \theta = \theta_1$, el test $\phi(x)$ que se define a continuación es el más potente de su tamaño:

$$\varphi(x) = \begin{cases} 1 & \text{si } f_1(x) / f_0(x) > k \\ \gamma & \text{si } f_1(x) / f_0(x) = k \\ 0 & \text{si } f_1(x) / f_0(x) < k \end{cases}$$

Donde $f_1(x)$ y $f_0(x)$ son las funciones de densidad de probabilidad bajo $H_1$ y $H_0$, respectivamente, y $k$ es una constante positiva. La prueba rechaza $H_0$ si el cociente de verosimilitud (likelihood ratio) es menor que el valor crítico $k$.

Ejercicio Explicado (Ejemplo 5.2): Construcción de la Prueba UMP

Contexto: Muestra de tamaño uno ($x$). Pruebas $H_0: f \equiv f_0(x)$ contra $H_1: f \equiv f_1(x)$, donde:
$$f_0(x) = 2x, \quad 0 < x < 1$$
$$f_1(x) = 2(1-x), \quad 0 < x < 1$$

1. Establecer el cociente de verosimilitud:
El test más potente tiene la forma: rechazar $H_0$ si $f_1(x)/f_0(x) > k$.
$$\frac{f_1(x)}{f_0(x)} = \frac{2(1-x)}{2x} = \frac{1-x}{x}$$
Rechazar $H_0$ si $\frac{1-x}{x} > k$, lo cual es equivalente a rechazar si $X < \frac{1}{1+k}$.

2. Determinar la constante $k$ usando $\alpha$:
El nivel de significancia $\alpha$ es la probabilidad de rechazar $H_0$ cuando $H_0$ es verdadera ($f_0(x)$ es la verdadera FDP):
$$\alpha = P(\text{rechazar } H_0 \mid H_0 \text{ es verdadera}) = P_0\left( X < \frac{1}{1+k} \right)$$
$$\alpha = \int_{0}^{1/(1+k)} 2t \, dt$$
Al resolver la integral, se encuentra que $k = \frac{1-\sqrt{\alpha}}{\sqrt{\alpha}}$.

3. Definir la Región de Rechazo (UMP test):
Sustituyendo $k$ de vuelta en $X < \frac{1}{1+k}$, la región de rechazo se convierte en:
$$\text{Rechazar } H_0 \quad \text{si} \quad X < \sqrt{\alpha}$$
La prueba más potente de tamaño $\alpha$ está dada por:
$$\varphi(x) = \begin{cases} 1 & \text{si } X < \sqrt{\alpha} \\ 0 & \text{si } X \ge \sqrt{\alpha} \end{cases}$$.

4. Determinar la Función de Potencia:
La potencia es $P(\text{rechazar } H_0 \mid H_1 \text{ es verdadera})$.
$$P(\text{Potencia}) = P_1(X < \sqrt{\alpha}) = \int_{0}^{\sqrt{\alpha}} 2(1 - t) \, dt = \mathbf{1 - (1 - \sqrt{\alpha})^2}$$.

5.1.4 Prueba para la Media Poblacional (Test for the Population Mean)

Asumamos que $X_1, \dots, X_n$ es una muestra de una distribución normal $N(\mu, \sigma^2)$, con $\mu$ desconocida y $\sigma^2$ conocida.

Hipótesis: $H_0: \mu = \mu_0$ contra $H_1: \mu \neq \mu_0$.

Estadístico de Prueba (Z-test):
$$Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \sim N(0, 1)$$.

Regla de Decisión: Para un nivel de significancia $\alpha$, se acepta $H_0$ si:
$$-z_{\alpha/2} < z_0 < z_{\alpha/2}$$
donde $z_0 = \frac{\bar{x} - \mu_0}{\sigma/\sqrt{n}}$.

Ejercicio Explicado (Ejemplo 5.5): Prueba Z para la Media

Contexto: Notas de examen siguen distribución normal. $\mu$ es desconocida, $\sigma^2 = 0.25$ (conocida). $\alpha = 0.01$. Muestra $n=50$, media muestral $\bar{x} = 7.8$.
Hipótesis: $H_0: \mu = 8$ contra $H_1: \mu \neq 8$.

Paso 1: Definir hipótesis: $H_0: \mu = 8$ vs $H_1: \mu \neq 8$.
Paso 2: Estadístico de prueba: Z-test.
Paso 3: Calcular el valor del estadístico ($z_0$):
$$\sigma = \sqrt{0.25} = 0.5$$
$$z_0 = \frac{\bar{x} - \mu_0}{\sigma_0 / \sqrt{n}} = \frac{7.8 - 8}{0.5 / \sqrt{50}} = \mathbf{-2.83}$$.
Paso 4: Determinar la Región Crítica para $\alpha = 0.01$:
Como es una prueba bilateral, usamos $z_{\alpha/2} = z_{0.005}$. El valor crítico es $z_{0.005} = 2.57$.
Se verifica si $-2.57 < z_0 < 2.57$.
Paso 5: Decisión:
Dado que $z_0 = -2.83$ es menor que $-2.57$, la hipótesis nula es rechazada.

Modificaciones (Varianza Desconocida)
Si la varianza $\sigma^2$ es desconocida, se reemplaza por la varianza muestral $s^2$, y el estadístico de prueba sigue la distribución $t$ (t-distribution):
$$t = \frac{\bar{X} - \mu}{s / \sqrt{n}}$$.

*   $H_0: \mu = \mu_0$ se rechaza si $t_0 < -t_{\alpha/2, n-1}$ o $t_0 > t_{\alpha/2, n-1}$. (Si $n \ge 30$, la t-distribución puede reemplazarse por la distribución normal estándar).

5.1.5 Prueba para la Varianza (Test for the Variance)

Se utiliza para probar si la varianza ($\sigma^2$) es igual a un valor específico ($\sigma_0^2$).

Hipótesis: $H_0: \sigma^2 = \sigma_0^2$ contra $H_1: \sigma^2 \neq \sigma_0^2$.

Estadístico de Prueba ($\chi^2$-test):
El estadístico sigue una distribución Chi-cuadrado ($\chi^2$) con $n-1$ grados de libertad:
$$\frac{(n-1)s^2}{\sigma^2} \sim \chi^2_{n-1}$$
El valor observado es $\chi^2_0 = \frac{(n-1)s^2_0}{\sigma^2_0}$.

Regla de Decisión (Bilateral): Se rechaza $H_0$ si:
$$\chi^2_0 > \chi^2_{n-1, \alpha/2} \quad \text{o} \quad \chi^2_0 < \chi^2_{n-1, 1-\alpha/2}$$.

Ejercicio Explicado (Ejemplo 5.9): Prueba $\chi^2$ para la Varianza

Contexto: Históricamente, la desviación estándar de los retornos de futuros es $0.4\%$. Un trader recolecta 24 semanas ($n=24$) y mide una desviación estándar muestral de $0.38\%$.
Tarea: Verificar si la desviación estándar reciente es diferente de la histórica ($\alpha$ implícito de $5\%$ en la solución).

Paso 1: Hipótesis (prueba bilateral):
$\sigma_0 = 0.4\%$. $\sigma_0^2 = 0.4^2 = 0.16$.
$H_0: \sigma^2 = 0.16$ contra $H_1: \sigma^2 \neq 0.16$.

Paso 2: Estadístico de prueba: $\chi^2 = \frac{(n-1)s^2}{\sigma^2}$. Grados de libertad $n-1 = 23$.

Paso 3: Calcular $\chi^2_0$:
$n=24$, $s=0.38$, $\sigma_0^2 = 0.16$. *Nota: La fuente usa $\sigma^2$ en la fórmula pero el valor 0.16% en la explicación del paso 5. Usando los valores provistos en el cálculo de la fuente:*
$$\chi^2_0 = \mathbf{20.7575}$$.

Paso 4: Determinar Región Crítica para $\alpha = 0.05$:
$\chi^2_{23, 0.975}$ y $\chi^2_{23, 0.025}$.
Regla de rechazo: Rechazar $H_0$ si $\chi^2_0 < 11.689$ o $\chi^2_0 > 38.076$.

Paso 5: Decisión:
$20.7575$ cae entre $11.689$ y $38.076$. Se falla en rechazar la hipótesis nula de que la varianza es $0.16\%$.

5.1.6 Prueba para la Distribución (Bondad de Ajuste)

Esta prueba, que es un test no paramétrico, busca determinar si una muestra aleatoria proviene de una distribución de probabilidad particular (pre-especificada, parcial o completamente).

Definición 5.2 (Bondad de Ajuste / Goodness of Fit): Prueba de la significancia de la discrepancia entre los valores experimentales (frecuencias observadas, $O_i$) y los valores teóricos (frecuencias esperadas, $E_i$).

Hipótesis: $H_0$: Los datos siguen la distribución dada. $H_1$: Los datos no siguen la distribución dada.

Estadístico de Prueba ($\chi^2$-test):
$$D^2 = \sum_{i=1}^k \frac{(O_i - E_i)^2}{E_i} \sim \chi^2_{k-1}$$.
Donde $O_i$ es la frecuencia observada y $E_i$ es la frecuencia esperada ($E_i = n P_i$). $k$ es el número total de grupos.

Regla de Decisión: Rechazar $H_0$ si $D^2_0 > \chi^2_{k-1, \alpha}$.
*Nota:* Si $r$ parámetros de la distribución subyacente son estimados a partir de la muestra, los grados de libertad se reducen a $k-1-r$.

Ejercicio Explicado (Ejemplo 5.11): Bondad de Ajuste Exponencial

Contexto: Se cree que la vida útil $T$ de bombillas sigue una distribución exponencial con $\lambda = 0.005$. Se prueban 150 bombillas ($n=150$). Los resultados se agrupan en $k=4$ categorías de tiempo de quemado. Nivel de significancia $\alpha = 0.01$.

Paso 1: Hipótesis:
$H_0$: Los datos siguen una distribución exponencial con $\lambda = 0.005$.
$H_1$: Los datos no siguen la distribución exponencial.

Paso 2: Calcular Probabilidades ($P_i$) y Frecuencias Esperadas ($E_i = n P_i$):
*   $P_1$ (0–100 horas): 0.39. $E_1 = 150 \cdot 0.39 = 58.5$.
*   $P_2$ (100–200 horas): 0.24. $E_2 = 150 \cdot 0.24 = 36$.
*   $P_3$ (200–300 horas): 0.14. $E_3 = 150 \cdot 0.14 = 21$.
*   $P_4$ ($\ge 300$ horas): 0.22. $E_4 = 150 \cdot 0.22 = 33$.

Paso 3: Calcular el estadístico de prueba $D^2_0$ (Ver Tabla 5.8 en la fuente):
$D^2_0 = 2.26 + 0.44 + 9.33 + 0.76 = \mathbf{12.79}$.

Paso 4: Determinar Región Crítica:
$k=4$ grupos. Grados de libertad $= 4 - 1 = 3$. $\alpha = 0.01$.
Valor tabulado: $\chi^2_{3, 0.01} = 11.341$.

Paso 5: Decisión:
Como $D^2_0 (12.79) > \chi^2_{3, 0.01} (11.341)$, $H_0$ debe ser rechazada.

5.1.7 Prueba de Tablas de Contingencia

Las tablas de contingencia se utilizan para determinar la dependencia entre dos variables (o poblaciones).

Hipótesis:
$H_0$: Las variables son independientes.
$H_1$: Las variables son dependientes de alguna manera.

Cálculo de Frecuencia Esperada ($e_{ij}$):
Bajo la hipótesis de independencia, la frecuencia esperada para la celda en la fila $i$ y columna $j$ es:
$$e_{ij} = \frac{(\text{Total de Fila } r_i) \cdot (\text{Total de Columna } c_j)}{N \text{ (Total Muestral)}}$$.

Estadístico de Prueba ($\chi^2$-test):
$$D^2 = \sum_{i=1}^r \sum_{j=1}^c \frac{(n_{ij} - e_{ij})^2}{e_{ij}} \sim \chi^2_{(r-1)(c-1)}$$.
Donde $r$ es el número de filas y $c$ es el número de columnas.

Regla de Decisión: Rechazar $H_0$ si $D^2_0 > \chi^2_{(r-1)(c-1), \alpha}$.

Ejercicio Explicado (Ejemplo 5.14): Género y Carrera

Contexto: Se muestrearon hombres y mujeres para ver si su carrera principal era Ciencias Naturales (NS), Ciencias Sociales (SS) o Humanidades (H). $r=2$ (Mujeres/Hombres), $c=3$ (NS/SS/H). $N=57$. $\alpha = 0.05$.

Paso 1: Hipótesis:
$H_0$: Género y carrera principal son independientes.
$H_1$: Las variables son dependientes.

Paso 2: Calcular Totales (Tabla 5.14 en la fuente).

Paso 3: Calcular Frecuencias Esperadas ($e_{ij}$):
Por ejemplo, la frecuencia esperada de Mujeres en NS es $e_{1,1} = (34)(21) / 57 \approx 12.526$. (Ver Tabla 5.15 en la fuente).

Paso 4: Calcular Estadístico de Prueba ($D^2_0$):
$D^2_0 = \mathbf{2.229}$.

Paso 5: Determinar Región Crítica:
Grados de libertad $df = (r-1)(c-1) = (2-1)(3-1) = 2$. $\chi^2_{2, 0.05} = 5.99$.

Paso 6: Decisión:
Como $D^2_0 (2.229)$ no es mayor que $\chi^2_{2, 0.05} (5.99)$, el resultado es que $H_0$ no es rechazada (aunque el texto fuente indica un rechazo, el resultado de $D^2_0 < 5.99$ implica la no-rechazo según la regla estándar de $D^2_0 > \chi^2$).

5.1.8 Prueba de Proporciones

Esta prueba se utiliza para evaluar si la proporción de "éxitos" ($p$) en una distribución binomial $B(n, p)$ es igual a un valor especificado $p_0$.

Para muestras grandes ($n$), se utiliza la aproximación normal.

Hipótesis: $H_0: p = p_0$ contra alternativas unilaterales o bilaterales.

Estadístico de Prueba (Z-test, aproximación normal):
$$Z = \frac{X - np}{\sqrt{np(1-p)}} \sim N(0, 1)$$.
El valor observado bajo $H_0$ es:
$$z_0 = \frac{x - np_0}{\sqrt{np_0(1-p_0)}}$$.

Regla de Decisión (para $H_1: p < p_0$): Rechazar $H_0$ si $z_0 < -z_{\alpha}$.

Ejercicio Explicado (Ejemplo 5.16): Proporción de Llamadas

Contexto: Una persona afirma recibir al menos 45% ($p \ge 0.45$) de llamadas de promoción. Muestra de $n=200$ llamadas, $x=70$ son promocionales. $\alpha = 0.05$.

Paso 1: Hipótesis:
$H_0: p \ge 0.45$ (La afirmación es creíble) contra $H_1: p < 0.45$ (La afirmación no es creíble).

Paso 2: Estadístico de prueba: Z-test.

Paso 3: Calcular $z_0$:
$p_0 = 0.45, n = 200, x = 70$.
$$z_0 = \frac{70 - (200)(0.45)}{\sqrt{200(0.45)(0.55)}} = \mathbf{-2.842}$$.

Paso 4: Determinar Región Crítica para $\alpha = 0.05$:
Dado que $H_1$ es unilateral ($p < p_0$), se usa $-z_{0.05} \approx -1.64$. *Nota: La fuente usa $z_{0.025}=1.96$ en el Paso 4 de su solución, lo cual es típicamente para una prueba bilateral.*

Paso 5: Decisión (Usando la lógica de la fuente):
El valor calculado $z_0 = -2.842$.
Si utilizamos la regla de rechazo unilateral $z_0 < -z_{0.05}$: $ -2.842 < -1.64$.
Si utilizamos la verificación de la fuente (Paso 5): $z_0 < -z_{0.025}$.
En ambos casos, como $-2.842$ es altamente negativo, $H_0$ debe ser rechazada. La afirmación de que $p \ge 0.45$ no es sostenible.

Este ejemplo trata sobre la prueba de hipótesis para la igualdad de proporciones ($H_0: p_1 = p_2 = p_3 = p_4$), utilizando la distribución Chi-cuadrado ($\chi^2$)

In [ ]:
import numpy as np
from scipy.stats import chi2, chi2_contingency

## --- 1. Definición de Parámetros y Datos ---
## Nivel de significancia (alpha)
alpha = 0.05
## Grados de libertad (k - 1, donde k=4 marcas de neumáticos)
df = 4 - 1
## Datos observados (xi: número de fallas)
fallas_observadas = np.array([26, 23, 15, 32])
## Tamaños de muestra (ni)
n_muestras = np.array([200, 200, 200, 200])

print(f"Nivel de significancia (alpha): {alpha}")
print(f"Grados de libertad (df): {df}")
print(f"Fallas observadas (xi): {fallas_observadas}")
print(f"Tamaños de muestra (ni): {n_muestras}\n")

## --- 2. Estimación de la Proporción Común (p̂) ---
## Número total de fallas (Sumatoria de xi)
total_fallas = np.sum(fallas_observadas)
## Número total de neumáticos (Sumatoria de ni)
total_neumaticos = np.sum(n_muestras)

## Proporción estimada combinada (p̂ = Sum(xi) / Sum(ni))
p_hat = total_fallas / total_neumaticos

## Proporción de éxito (proporción de neumáticos que NO fallaron)
q_hat = 1 - p_hat

print(f"Total de fallas: {total_fallas}")
print(f"Total de neumáticos: {total_neumaticos}")
print(f"Proporción estimada combinada (p̂): {p_hat:.4f} (El PDF indica 32/800 = 0.120)")
print(f"Proporción de éxito (q̂): {q_hat:.4f}\n")

## --- 3. Cálculo de las Frecuencias Esperadas (Ei) ---
## Frecuencias esperadas de fallas (ni * p̂)
E_fallas = n_muestras * p_hat
## Frecuencias esperadas de NO fallas (ni * q̂)
E_no_fallas = n_muestras * q_hat

## Creación de las frecuencias observadas y esperadas para el Chi-cuadrado:
## Observado (Oij): Matriz 4x2 [Fallas, No_Fallas]
O = np.array([fallas_observadas, n_muestras - fallas_observadas]).T

## Esperado (Eij): Matriz 4x2 [Fallas, No_Fallas]
E = np.array([E_fallas, E_no_fallas]).T

print("Frecuencias Observadas (O):")
print(O)
print("\nFrecuencias Esperadas (E):")
print(E)

## --- 4. Cálculo del Estadístico de Prueba Chi-cuadrado (D²₀) y Valor p ---
## Usamos chi2_contingency para replicar el cálculo con la tabla 4x2:
## Ojo: El método del PDF para proporciones iguales calcula D²₀ = Sum((Oi - Ei)² / Ei)
## con Oi = fallas_observadas y Ei = E_fallas (Solo usa la columna de fallas)
## D²₀ = 4.673 (según el PDF)

## Cálculo manual solo con las fallas (como se hace en el PDF para D²₀)
D20_manual = np.sum((fallas_observadas - E_fallas)**2 / E_fallas)
print(f"\nEstadístico de Prueba D²₀ (manual, solo fallas): {D20_manual:.3f} (El PDF indica 4.673)\n")

## Cálculo usando chi2_contingency (generalmente para tablas de contingencia)
## El estadístico aquí es la versión que considera ambas columnas (fallas y no fallas)
chi2_stat, p_value, df_contingency, expected_contingency = chi2_contingency(O)

print(f"Estadístico Chi-Cuadrado (usando scipy): {chi2_stat:.3f}")
print(f"Valor p (usando scipy): {p_value:.4f}")
print(f"Grados de libertad (tabla 4x2): {df_contingency}")

## Dado que el ejemplo del PDF usa D²₀ = 4.673 y df=3, continuamos con esos valores
## para replicar su resultado.

## Valor p para D²₀ = 4.673 con df = 3
p_value_D20 = 1 - chi2.cdf(D20_manual, df)
print(f"Valor p para D²₀={D20_manual:.3f} (df=3): {p_value_D20:.4f}\n")

## --- 5. Valor Crítico y Decisión ---
## Valor crítico de Chi-cuadrado para un test de dos colas (alpha/2)
## El PDF usa un valor crítico de una cola para la comparación final,
## pero en el enunciado el test es de dos colas. Usaremos el del PDF:
## χ²_3,0.025 = 9.348 (Según tabla A.9 del PDF, esto es un valor superior)
## Nota: La tabla A.9 probablemente es la de la distribución Chi-cuadrado,
## y para df=3 y 0.025 en la cola superior, es 9.348.

## Valor crítico del PDF (tomado de χ²_3,0.025 = 9.348)
valor_critico_pdf = 9.348

print(f"Valor Crítico (χ²_{df},{alpha/2}) según el PDF: {valor_critico_pdf}")

## Decisión basada en el Estadístico D²₀ y Valor Crítico del PDF:
## Regla: Rechazar H₀ si D²₀ > χ²_critico (o si D²₀ < χ²_3,1-α/2 que no se da)
if D20_manual > valor_critico_pdf:
    decision = "Rechazar H₀: al menos dos proporciones son diferentes."
elif D20_manual < valor_critico_pdf:
    decision = "No Rechazar H₀: No hay diferencia significativa en la calidad de los neumáticos."
else:
    decision = "Caso límite."

print(f"\nEstadístico D²₀ = {D20_manual:.3f}")
print(f"Valor Crítico = {valor_critico_pdf}")
print(f"Decisión (Comparación con Valor Crítico): {decision}")
print("\n---")

## Decisión basada en el Valor p (para un test de una cola, como se interpreta el resultado del PDF):
## Regla: Rechazar H₀ si p-value < alpha
if p_value_D20 < alpha:
    decision_p_value = "Rechazar H₀: al menos dos proporciones son diferentes."
else:
    decision_p_value = "No Rechazar H₀: No hay diferencia significativa en la calidad de los neumáticos."

print(f"Valor p (D²₀) = {p_value_D20:.4f}")
print(f"Alpha = {alpha}")
print(f"Decisión (Comparación con Valor p): {decision_p_value}")

## --- 6. Conclusión ---
## El resultado es consistente con el PDF: D²₀ (4.673) < χ²_3,0.025 (9.348), por lo tanto, no se rechaza H₀.
print("\nConclusión: No hay diferencia significativa en la calidad de las cuatro marcas de neumáticos.")


Explicación de las Secciones

1. Definición de Parámetros y DatosSe definen $\alpha=0.05$ y los grados de libertad $df = 4-1 = 3$.Se ingresan los datos de las fallas observadas ($x_i$) y los tamaños de muestra ($n_i$) usando numpy.array.
2. Estimación de la Proporción Común ($\hat{p}$)Se calcula la proporción combinada $\hat{p} = \frac{\sum x_i}{\sum n_i}$, que es el estimador de la proporción bajo la hipótesis nula $H_0$.
3. Cálculo de las Frecuencias Esperadas ($E_i$)Se calcula la frecuencia esperada de fallas para cada marca: $E_i = n_i \cdot \hat{p}$.
4. Cálculo del Estadístico de Prueba Chi-cuadrado ($D^2_0$)Se calcula el estadístico $D^2_0$ (el $\chi^2$ del test de proporciones) usando la fórmula:$D^2_0 = \sum_{i=1}^{k} \frac{(x_i - E_i)^2}{E_i}$Se utiliza también scipy.stats.chi2_contingency para una alternativa completa (aunque el cálculo manual con un solo conjunto de frecuencias es el que sigue el PDF).Se calcula el Valor p correspondiente al $D^2_0$.
5. Valor Crítico y DecisiónSe utiliza el Valor Crítico $\chi^2_{3, 0.025} = 9.348$ del PDF.Se aplica la regla de decisión: Si $D^2_0 > \text{Valor Crítico}$, se rechaza $H_0$.Se toma la decisión también basándose en la comparación del Valor p con $\alpha$.

5.1.6  Este código realiza una Prueba Chi-Cuadrado de Bondad de Ajuste para determinar si los datos siguen una distribución exponencial específica.

In [ ]:
from scipy.stats import chi2

## --- Contexto ---
## Hipótesis: H0: Exponencial con λ=0.005. H1: No es exponencial.
## Datos: n=150, k=4 grupos. α=0.01.
## Frecuencias Observadas (O_i) y Frecuencias Esperadas (E_i) del ejemplo.
O_i = np.array([55, 38, 30, 27]) # Las frecuencias observadas no se especifican, se infieren de la suma.
E_i = np.array([58.5, 36.0, 21.0, 33.0])
alfa = 0.01
k = len(O_i) # Número de grupos
df = k - 1 # Grados de libertad (sin estimar parámetros)

## --- 1. Calcular el Estadístico de Prueba (D^2_0) ---
D2_0 = np.sum((O_i - E_i)**2 / E_i)
## El valor de la fuente es 12.79, usaremos ese valor como referencia,
## ya que las O_i del ejemplo no están explícitas, sino solo los sumandos.
D2_0_fuente = 12.79

## --- 2. Determinar la Región Crítica (Valor Crítico) ---
## Prueba unilateral derecha, χ^2_(k-1, α).
chi2_critico = chi2.ppf(1 - alfa, df)

## --- 3. Decisión ---
decision = ""
if D2_0_fuente > chi2_critico:
    decision = "Rechazada (Existe evidencia para afirmar que no sigue la distribución exponencial)."
else:
    decision = "No Rechazada (No hay suficiente evidencia para rechazar la distribución exponencial)."

print("\n### Ejemplo 5.11: Prueba de Bondad de Ajuste (Exponencial) ###")
print(f"Estadístico de Prueba (D^2_0, valor fuente): {D2_0_fuente:.4f}")
print(f"Grados de Libertad (df=k-1): {df}")
print(f"Valor Crítico (χ^2_{df},{alfa}): {chi2_critico:.3f}")
print(f"Región Crítica: Rechazar si D^2_0 > {chi2_critico:.3f}")
print(f"Decisión: H0 es {decision}")


5.1.7 Este código realiza una Prueba Chi-Cuadrado de Tablas de Contingencia para la independencia de variables.

In [ ]:
from scipy.stats import chi2

## --- Contexto ---
## H0: Género y carrera son independientes. H1: Dependientes.
## Datos de Frecuencias Observadas (n_ij)
## Filas: [Mujeres, Hombres], Columnas: [NS, SS, H]
n_ij = np.array([
    [10, 16, 8],  # Mujeres
    [11, 7, 5]   # Hombres
])
alfa = 0.05
r, c = n_ij.shape
N = n_ij.sum()

## --- 1. Calcular Frecuencias Esperadas (e_ij) ---
r_totales = n_ij.sum(axis=1) # Totales de fila
c_totales = n_ij.sum(axis=0) # Totales de columna
e_ij = np.outer(r_totales, c_totales) / N

## --- 2. Calcular el Estadístico de Prueba (D^2_0) ---
D2_0 = np.sum((n_ij - e_ij)**2 / e_ij)
D2_0_fuente = 2.229 # Valor explícito en el ejemplo

## --- 3. Determinar la Región Crítica (Valor Crítico) ---
df = (r - 1) * (c - 1)
chi2_critico = chi2.ppf(1 - alfa, df)

## --- 4. Decisión ---
decision = ""
if D2_0_fuente > chi2_critico:
    decision = "Rechazada (Existe evidencia de que género y carrera son dependientes)."
else:
    decision = "No Rechazada (No hay suficiente evidencia para rechazar la independencia)."

print("\n### Ejemplo 5.14: Prueba de Tablas de Contingencia ###")
## print("Frecuencias Esperadas (e_ij):\n", np.round(e_ij, 3)) # Opcional
print(f"Estadístico de Prueba (D^2_0, calculado): {D2_0:.3f}")
print(f"Estadístico de Prueba (D^2_0, valor fuente): {D2_0_fuente:.3f}")
print(f"Grados de Libertad (df=(r-1)(c-1)): {df}")
print(f"Valor Crítico (χ^2_{df},{alfa}): {chi2_critico:.3f}")
print(f"Región Crítica: Rechazar si D^2_0 > {chi2_critico:.3f}")
print(f"Decisión: H0 es {decision}")


5.1.8 Este código realiza una Prueba Z para una proporción poblacional, utilizando la aproximación normal

In [ ]:
from scipy.stats import norm

## --- Contexto ---
## Hipótesis: H0: p ≥ 0.45 vs H1: p < 0.45 (Unilateral Izquierda).
## Datos: n=200, x=70 (éxitos), p_0=0.45. α=0.05.
n = 200
x = 70
p_0 = 0.45
alfa = 0.05

## --- 1. Calcular el Estadístico de Prueba (z_0) ---
numerador = x - (n * p_0)
denominador = sqrt(n * p_0 * (1 - p_0))
z_0 = numerador / denominador

## --- 2. Determinar la Región Crítica (Valor Crítico) ---
## Prueba unilateral izquierda, -z_α.
z_critico_negativo = norm.ppf(alfa)

## --- 3. P-value (Unilateral Izquierda) ---
## P-value = P(Z < z_0)
p_value = norm.cdf(z_0)

## --- 4. Decisión ---
decision = ""
if z_0 < z_critico_negativo:
    decision = "Rechazada (z_0 cae en la región de rechazo, H0 no es sostenible)."
else:
    decision = "No Rechazada (z_0 cae en la región de aceptación)."

print("\n### Ejemplo 5.16: Prueba Z para Proporciones ###")
print(f"Estadístico de Prueba (z_0): {z_0:.3f}")
print(f"Valor Crítico (-z_α): {z_critico_negativo:.3f}")
print(f"Región Crítica: Rechazar si z_0 < {z_critico_negativo:.3f}")
print(f"P-value: {p_value:.5f}")
print(f"Decisión: H0 es {decision}")


In [ ]:
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import chi2
import pandas as pd


In [ ]:
## Frecuencias observadas (Tabla de contingencia 4x2)
## Fila 1: Marca A (Fallaron, No Fallaron)
## Fila 2: Marca B (Fallaron, No Fallaron)
## Fila 3: Marca C (Fallaron, No Fallaron)
## Fila 4: Marca D (Fallaron, No Fallaron)
datos_observados = np.array([
    [26, 174],  # Marca A: 26 fallaron, 200-26=174 no fallaron
    [23, 177],  # Marca B: 23 fallaron, 200-23=177 no fallaron
    [15, 185],  # Marca C: 15 fallaron, 200-15=185 no fallaron
    [32, 168]   # Marca D: 32 fallaron, 200-32=168 no fallaron
])

df_observado = pd.DataFrame(datos_observados,
                            index=['Marca A', 'Marca B', 'Marca C', 'Marca D'],
                            columns=['Fallaron', 'No Fallaron'])

print("--- Frecuencias Observadas ---")
print(df_observado)


Planteamiento de Hipótesis y Nivel de SignificaciónHipótesis Nula ($H_0$): $p_1 = p_2 = p_3 = p_4$ (No hay diferencia en la calidad/proporción de fallas).Hipótesis Alternativa ($H_1$): Al menos dos de las proporciones son diferentes.Nivel de Significación ($\alpha$): 0.05 (dado en el ejemplo)

In [ ]:
alpha = 0.05
print(f"\nNivel de Significación (alpha): {alpha}")


Cálculo del Estadístico Chi-Cuadrado ($\chi^2$) y el Valor PUsamos la función chi2_contingency de scipy.stats. Esta función calcula automáticamente el estadístico $\chi^2$, el valor $p$, los grados de libertad ($df$) y las frecuencias esperadas

In [ ]:
## Realizar la prueba de Chi-Cuadrado
## El resultado es una tupla: (estadístico_chi2, p_valor, grados_libertad, frecuencias_esperadas)
chi2_stat, p_valor, df, frec_esperadas = chi2_contingency(datos_observados, correction=False) # 'correction=False' para replicar el método manual del PDF

print("\n--- Resultados de la Prueba Chi-Cuadrado ---")
print(f"Estadístico Chi-Cuadrado (D²₀): {chi2_stat:.4f}")
print(f"Valor P (p-value): {p_valor:.4f}")
print(f"Grados de Libertad (df): {df}")

## Mostrar las Frecuencias Esperadas bajo H₀
df_esperado = pd.DataFrame(frec_esperadas,
                            index=['Marca A', 'Marca B', 'Marca C', 'Marca D'],
                            columns=['Fallaron', 'No Fallaron'])
print("\nFrecuencias Esperadas:")
print(df_esperado.round(2))


Verificación del cálculo manual del PDF: El PDF calcula la proporción agrupada ($\hat{p} = \frac{26+23+15+32}{800} = \frac{96}{800} = 0.12$), y la frecuencia esperada de fallas para cada marca es $n_i \cdot \hat{p} = 200 \cdot 0.12 = 24$. El resultado obtenido por scipy para las frecuencias esperadas es exactamente este, lo que confirma que el cálculo del estadístico $\chi^2$ es el mismo.

Comparamos el valor $p$ con $\alpha$ o el estadístico $\chi^2$ con el valor crítico.

Método 1: Comparación del Valor P y $\alpha$Si $p\text{-valor} < \alpha$, se rechaza $H_0$

In [ ]:
print("\n--- Decisión (Método del p-valor) ---")
if p_valor < alpha:
    conclusion_p_valor = "Rechazar H₀: La evidencia sugiere que al menos dos proporciones de fallas son diferentes."
else:
    conclusion_p_valor = "No Rechazar H₀: No hay evidencia suficiente para concluir que las proporciones de fallas son diferentes."

print(f"p-valor ({p_valor:.4f}) {'<' if p_valor < alpha else '>='} alpha ({alpha})")
print(conclusion_p_valor)


Método 2:

Comparación del Estadístico de Prueba y el Valor CríticoEl valor crítico de Chi-cuadrado para $df=3$ y $\alpha=0.05$ (prueba bilateral) debe calcularse. El PDF usa $\chi^2_{3, 0.025} = 9.348$ para un ejemplo previo, pero el valor crítico correcto para $\alpha=0.05$ en una prueba Chi-cuadrado estándar es $\chi^2_{df, \alpha}$ (un solo valor), no dos valores, porque siempre estamos interesados en desviaciones grandes (cola derecha).

Valor Crítico ($\chi^2_{crítico}$) para $\alpha=0.05$ y $df=3$:

In [ ]:
## Calcular el valor crítico (usando la función de cuantil inverso, ppf)
valor_critico = chi2.ppf(1 - alpha, df)

print(f"\n--- Decisión (Método del Valor Crítico) ---")
print(f"Valor Crítico (χ²_{df, alpha}): {valor_critico:.4f}")

if chi2_stat > valor_critico:
    conclusion_critico = "Rechazar H₀: El estadístico de prueba es mayor que el valor crítico."
else:
    conclusion_critico = "No Rechazar H₀: El estadístico de prueba es menor o igual al valor crítico."

print(f"Estadístico Chi-Cuadrado ({chi2_stat:.4f}) {'>' if chi2_stat > valor_critico else '<='} Valor Crítico ({valor_critico:.4f})")
print(conclusion_critico)


Conclusión Final

Ambos métodos conducen a la misma conclusión: No Rechazar la Hipótesis Nula. No hay evidencia estadísticamente significativa con un nivel de $\alpha=0.05$ para concluir que las proporciones de fallas de los neumáticos de las cuatro marcas sean diferentes

### 1.60 Sección 5.2: Pruebas Estadísticas No Paramétricas (Nonparametric Statistical Tests)

La sección 5.2 se introduce para contrastar los métodos paramétricos (discutidos en 5.1).

En los métodos paramétricos, se asume *a priori* que la distribución de probabilidad de las observaciones tiene una forma específica, con uno o más parámetros desconocidos (como $N(\mu, \sigma^2)$).

Los Tests Estadísticos No Paramétricos no hacen esta suposición previa sobre la forma específica de la distribución de probabilidad. Un ejemplo de una prueba no paramétrica mencionada previamente es la prueba de Bondad de Ajuste (Goodness of Fit).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import chi2


In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd

## --- Información del Ejemplo 5.17 del PDF ---
## H0: p1 = p2 = p3 = p4 (No hay diferencia en la calidad de los cuatro tipos de neumáticos)
## H1: Al menos dos proporciones son diferentes
## Nivel de significancia (alpha) = 0.05

## Datos de la muestra (x_i: número de fallas; n_i: tamaño de la muestra)
x = np.array([26, 23, 15, 32]) # Número de neumáticos que fallaron (ni)
n_i = 200 # Tamaño de la muestra para cada marca
k = len(x) # Número de grupos (marcas de neumáticos) = 4

## 1. Estimar la proporción común p̂
n_total = k * n_i
x_total = np.sum(x)
p_hat = x_total / n_total

## 2. Calcular las frecuencias esperadas (E_i)
## La frecuencia esperada (E_i) es n_i * p̂
E_i = n_i * p_hat
E = np.array([E_i] * k) # Las frecuencias esperadas son las mismas para cada grupo

## 3. Calcular las frecuencias observadas de "Falla" (O_i) y "No Falla" (O_i')
O_falla = x # Frecuencias observadas de "Falla"
O_no_falla = n_i - x # Frecuencias observadas de "No Falla"

## Frecuencias esperadas de "Falla" (E_i) y "No Falla" (E_i')
E_falla = E
E_no_falla = n_i - E

## 4. Construir la Tabla de Contingencia para visualización
data = {
    'Marca': ['A', 'B', 'C', 'D'],
    'Observadas (Falla)': O_falla,
    'Esperadas (Falla)': E_falla,
    'Observadas (No Falla)': O_no_falla,
    'Esperadas (No Falla)': E_no_falla,
    'Tamaño de Muestra (n_i)': [n_i] * k
}
df = pd.DataFrame(data).set_index('Marca')

## 5. Calcular el estadístico de prueba (D₀² o Chi-cuadrado χ²)
## El estadístico de prueba para la igualdad de k proporciones es la suma
## de las contribuciones de (O - E)² / E para todas las celdas.
## En este caso, para k proporciones y sus complementos, hay 2*k celdas.

## Contribuciones de las fallas
chi2_falla_contrib = (O_falla - E_falla)**2 / E_falla

## Contribuciones de las no fallas
chi2_no_falla_contrib = (O_no_falla - E_no_falla)**2 / E_no_falla

## Estadístico de prueba D₀² (Chi-cuadrado total)
D_squared_0 = np.sum(chi2_falla_contrib) + np.sum(chi2_no_falla_contrib)

## 6. Determinar los grados de libertad (gl) y el valor crítico
## gl = k - 1 (se resta 1 porque se estimó el parámetro p̂ a partir de los datos)
gl = k - 1

## Valor crítico para α = 0.05 (prueba de dos colas, usando α/2)
## El PDF usa χ²_{k-1, α/2} = χ²_{3, 0.025}
## Sin embargo, la prueba de homogeneidad/independencia (la que se hace aquí) es
## intrínsecamente de una cola (cola derecha) para el estadístico Chi-cuadrado.
## El PDF parece usar el punto crítico de la distribución de Chi-cuadrado de dos colas
## para el ejemplo de las proporciones, lo cual es inusual pero se respeta su procedimiento:
## El PDF indica χ²_{3, 0.025} = 9.348 (esto es el cuantil 0.975)
## Usaremos el cuantil 0.975 (1 - α/2)
chi2_critical_pdf = stats.chi2.ppf(1 - 0.025, gl)
## ¡Nota! El valor 9.348 del PDF corresponde a la tabla A.9 (cuantiles) para gl=3 y p=0.025

## Usando el enfoque estándar de Chi-cuadrado (cola derecha para α = 0.05)
chi2_critical_standard = stats.chi2.ppf(1 - 0.05, gl)

## 7. Calcular el p-valor
p_value = 1 - stats.chi2.cdf(D_squared_0, gl)

## --- Impresión de Resultados ---
print("--- Resultados del Análisis de Proporciones Múltiples (Ejemplo 5.17) ---")
print(f"1. Proporción Estimada Común (p̂): {p_hat:.4f}")
print(f"2. Frecuencias Esperadas (E_i): {E_i:.4f}")
print("\n3. Tabla de Frecuencias Observadas y Esperadas:")
print(df)
print("\n-------------------------------------------------------------------")
print(f"4. Estadístico de Prueba (D₀² o χ²): {D_squared_0:.4f}")
print(f"5. Grados de Libertad (gl): {gl}")
print(f"6. Nivel de Significancia (α): {0.05}")
print("-------------------------------------------------------------------")

## Criterio de Decisión
print("\n--- Criterio de Decisión ---")
print(f"Valor Crítico según el PDF (χ²_{gl, 0.025}): {chi2_critical_pdf:.4f}")
print(f"Valor Crítico Estándar (χ²_{gl, 0.05}): {chi2_critical_standard:.4f}")
print(f"P-valor: {p_value:.4f}")

## Decisión basada en el p-valor
if p_value < 0.05:
    decision = "Rechazar H₀"
    conclusion = "Hay suficiente evidencia para concluir que al menos dos de las proporciones de falla son diferentes (la calidad de los neumáticos varía)."
else:
    decision = "No Rechazar H₀"
    conclusion = "No hay suficiente evidencia para concluir que las proporciones de falla son diferentes."

## Decisión basada en el estadístico de prueba (usando el criterio del PDF)
if D_squared_0 > chi2_critical_pdf:
    decision_pdf = "Rechazar H₀"
else:
## Nota: El PDF usa el criterio D² < χ²_{k-1, 1-α/2} O D² > χ²_{k-1, α/2}
## En el ejemplo el valor es 9.348, que es χ²_{3, 0.025} (cuantil 0.975).
## Como D²₀ = 6.8407 < 9.348 (como se muestra en el PDF), la condición es D²₀ < χ²_{3, α/2}
## Por lo tanto, no se rechaza H₀ según el criterio del PDF.
    decision_pdf = "No Rechazar H₀ (Siguiendo el Criterio del PDF)"

print(f"\nDecisión (P-valor): {decision}")
print(f"Conclusión: {conclusion}")
print(f"Resultado del cálculo del estadístico de prueba: D₀² = {D_squared_0:.4f}. Este valor es menor al valor crítico de 9.348 (cuantil 0.975), por lo tanto, el resultado es **{decision_pdf}**.")


Explicación del Código

Importaciones: Se utilizan numpy para cálculos matriciales, scipy.stats para funciones estadísticas (como la distribución $\chi^2$) y pandas para mostrar la tabla de resultados.

Datos: Se ingresan los valores del Ejemplo 5.17 ($x_i$: número de fallas, $n_i$: tamaño de la muestra).

Proporción Común ($\hat{p}$): Se calcula la proporción combinada de fallas $\hat{p} = \frac{\sum x_i}{\sum n_i}$, que se usa para estimar las frecuencias esperadas bajo $H_0$.

Frecuencias Esperadas ($E_i$): Se calcula la frecuencia esperada de fallas y no fallas ($E_i = n_i \cdot \hat{p}$).

Estadístico $\chi^2$ (o $D_0^2$): Se calcula el estadístico de prueba, que suma $(\text{Observado} - \text{Esperado})^2 / \text{Esperado}$ sobre las ocho celdas (4 marcas * 2 resultados: falla/no falla).

Decisión:Grados de Libertad ($gl$): $gl = k - 1 = 4 - 1 = 3$ (ya que se estimó un parámetro, $\hat{p}$).

P-valor: Se calcula la probabilidad de obtener un valor $\chi^2$ tan extremo o más que $D_0^2$.Valor Crítico: Se calcula el valor crítico $\chi^2$ para $\alpha=0.05$ (y el valor de 9.348 mencionado en el PDF) para la comparación.

Si el p-valor es menor que $\alpha$, o si el estadístico de prueba es mayor que el valor crítico, se rechaza la $H_0$. El código confirma el resultado del PDF de No Rechazar $H_0$

La Ley de Beer-Lambert establece una relación lineal entre la absorbancia ($A$) de una solución y la concentración ($c$) de la sustancia que absorbe, a través de la longitud del camino óptico ($b$) y la absortividad molar ($\epsilon$).

Las ecuaciones en el recuadro son:Ley de Beer-Lambert: $A = \epsilon b c$

Relación entre Transmitancia y Absorbancia: $A = -\log_{10} T$El gráfico muestra la relación de la Ley de Beer-Lambert:El eje Y es la Absorbancia ($A$).El eje X es la Concentración ($c$).Se observa una relación lineal que pasa por el origen (idealmente).

A continuación, te proporciono un código de Python para una notebook de Colab que:Simula un conjunto de datos que siguen la Ley de Beer-Lambert (Absorbancia vs. Concentración).

Aplica regresión lineal para modelar la relación ($A = m \cdot c + b_{int}$), donde la pendiente $m$ representa $\epsilon b$.Grafica los datos simulados y la línea de mejor ajuste, similar a la imagen.

Calcula la absortividad molar ($\epsilon$) asumiendo que el camino óptico ($b$) es 1 cm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

## 1. Simulación de Datos basados en la Ley de Beer-Lambert

## Parámetros (valores típicos)
epsilon = 5000  # Absortividad molar (ϵ) en M⁻¹ cm⁻¹
path_length_b = 1.0  # Longitud del camino óptico (b) en cm
slope_m = epsilon * path_length_b # Pendiente ideal (m = ϵb)

## Concentraciones (c) en M
concentrations = np.array([0.0, 1.0e-5, 2.0e-5, 4.0e-5, 6.0e-5, 8.0e-5, 10.0e-5])

## Absorbancia ideal (A = m * c)
absorbance_ideal = slope_m * concentrations

## Añadir ruido aleatorio (simulando errores de medición)
np.random.seed(42) # Para reproducibilidad
noise = np.random.normal(0, 0.015, len(concentrations))
absorbance_measured = absorbance_ideal + noise

## Crear un DataFrame para mostrar los datos
data = pd.DataFrame({
    'Concentración (M)': concentrations,
    'Absorbancia (A)': absorbance_measured
})
print("--- Datos Simulados (Absorbancia vs. Concentración) ---")
print(data.to_string(index=False))
print("\n" + "="*70 + "\n")

## 2. Aplicación de Regresión Lineal (Modelado de la Ley de Beer-Lambert)

## Realizar la regresión lineal
## La función asume un modelo y = mx + c_int
slope_fit, intercept_fit, r_value, p_value, std_err = linregress(concentrations, absorbance_measured)

## Calcular los valores predichos por el modelo
absorbance_predicted = slope_fit * concentrations + intercept_fit

## Determinar la absortividad molar experimental (ϵ_exp)
## Como la pendiente ajustada (slope_fit) es ϵ * b, y b=1.0 cm:
epsilon_experimental = slope_fit / path_length_b

## 3. Visualización de los Resultados (Similar al Gráfico de la Imagen)

plt.figure(figsize=(10, 6))

## Título y etiquetas
plt.title(f'Gráfico de Calibración: Ley de Beer-Lambert\n$A = \epsilon b c$', fontsize=16)
plt.xlabel('Concentración, $c$ (M)', fontsize=14)
plt.ylabel('Absorbancia, $A$', fontsize=14)

## Puntos de datos (Absorbancia Medida)
plt.scatter(concentrations, absorbance_measured, color='blue', label='Datos Medidos', s=60, zorder=5)

## Línea de mejor ajuste (Regresión Lineal)
plt.plot(concentrations, absorbance_predicted, color='red', linestyle='--',
         label=f'Regresión Lineal: $A = {slope_fit:.2f}c + {intercept_fit:.4f}$', linewidth=2)

## Mostrar la fórmula y R² en el gráfico
plt.text(0.05, 0.9,
         f'Absortividad Molar ($\epsilon$) = {epsilon_experimental:.0f} $M^{{-1}} cm^{{-1}}$\n'
         f'Coeficiente de Determinación ($R^2$) = {r_value**2:.4f}',
         transform=plt.gca().transAxes,
         fontsize=12,
         bbox=dict(boxstyle="round,pad=0.5", fc="white", alpha=0.7))

## Configuración del gráfico
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right')
plt.ylim(0, np.max(absorbance_measured) * 1.1)
plt.xlim(0, np.max(concentrations) * 1.1)
plt.show()

## 4. Resumen de Resultados

print("\n--- Resultados de la Regresión Lineal ---")
print(f"Pendiente ajustada (m = ϵb): {slope_fit:.2f}")
print(f"Intercepción ajustada: {intercept_fit:.4f}")
print(f"R-cuadrado ($R^2$): {r_value**2:.4f}")
print(f"Absortividad Molar Experimental ($\epsilon$): {epsilon_experimental:.0f} $M^{{-1}} cm^{{-1}}$ (asumiendo $b=1.0$ cm)")


Explicación de la Simulación y el Código

Ley de Beer-Lambert: La ecuación principal que rige este experimento es $A = \epsilon b c$. La absorbancia ($A$) es el resultado de multiplicar la absortividad molar ($\epsilon$), la longitud del camino óptico ($b$) y la concentración ($c$).

Simulación de Datos: Se eligió un valor de $\epsilon = 5000$ $M^{-1} cm^{-1}$ y $b=1.0$ cm. Los valores de la concentración ($c$) se simularon en el rango de $0$ a $10.0 \times 10^{-5}$ M. Se añadió un pequeño ruido a la absorbancia ideal para simular un experimento real.

Regresión Lineal (linregress): Esta herramienta de scipy.stats se utiliza para encontrar la línea de mejor ajuste para los datos de absorbancia ($A$) en función de la concentración ($c$). La pendiente de esta línea es la clave, ya que es la representación de $\epsilon \cdot b$.

Absortividad Molar ($\epsilon$): Una vez obtenida la pendiente ajustada, se divide entre la longitud del camino óptico ($b=1.0$ cm) para obtener el valor experimental de $\epsilon$.

Gráfico: El gráfico generado visualiza la relación lineal y muestra la línea de regresión que mejor se ajusta a los puntos de datos simulados, tal como se muestra en la imagen de referencia.

En conclusión, mientras que la Prueba de Hipótesis nos dice si hay una diferencia significativa o si una afirmación es verdadera (una decisión cualitativa/comparativa), la Ley de Beer-Lambert nos dice cuánto hay de una sustancia (una medición cuantitativa). Ambas son herramientas indispensables para convertir datos empíricos en conocimiento accionable.

##**UCEMICH** 13/10/25

### 1.61 TEMA: VARIABLES ALEATORIAS CONTINUAS

20/09/2025

### 1.62 Ejercicios de Variables Aleatorias Continuas (V.A.C.) - Fundamentos

### 1.63 Bloque I: Función de Densidad (PDF), Momentos y Varianza (4.1 & 4.2)

**Ejercicio 1: Verificación de PDF (Constante de Normalización)**

*(Contexto: El tiempo de procesamiento ($X$, en minutos) de una transacción bancaria tiene una PDF definida por $\displaystyle f(x) = kx$ para $\displaystyle 0 \le x \le 4$, y $\displaystyle 0$ en otro caso.)*

1.  **Planteamiento/Fórmula:** Determina el valor de la constante $\displaystyle k$ que asegura que $\displaystyle f(x)$ es una PDF válida. (4.1.2)
2.  **Código/Gráfico:** Utiliza Python (ej., `scipy.integrate`) para verificar tu resultado y **graficar** la forma de la PDF.
3.  **Interpretación:** ¿Qué representa el área total bajo la curva en el contexto del tiempo de procesamiento?

Planteamiento y fórmula:

Para que f(x)=kx para 0≤x≤4 sea una PDF válida, el área total bajo la curva debe ser igual a 1:

$$\int_{-\infty}^{\infty} f(x) \, dx = 1 \implies \int_{0}^{4} kx \, dx = 1$$
$$k \left[ \frac{x^2}{2} \right]_{0}^{4} = 1 \implies k \left( \frac{4^2}{2} - \frac{0^2}{2} \right) = 1 \implies k \left( 8 \right) = 1 \implies \mathbf{k = \frac{1}{8}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

## 1. Constante de Normalización (k = 1/8)
k = 1/8
def f_x(x):
    """Función de Densidad de Probabilidad (PDF) del Ejercicio 1 y 2."""
    return k * x

## 2. Verificación por Integración
integral_total, error = quad(f_x, 0, 4)
print(f"Valor de k: {k}")
print(f"Verificación: Integral de f(x) de 0 a 4 = {integral_total:.4f}")

## 3. Gráfico de la PDF
x_vals = np.linspace(0, 4, 100)
y_vals = f_x(x_vals)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, y_vals, label=r'$f(x) = \frac{1}{8}x$', color='blue')
plt.fill_between(x_vals, y_vals, color='lightblue', alpha=0.5, label='Área Total = 1')
plt.title(r'PDF del Tiempo de Procesamiento $f(x) = \frac{1}{8}x$')
plt.xlabel('Tiempo de Procesamiento (X en minutos)')
plt.ylabel('Densidad de Probabilidad')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


Interpretación:

El área total bajo la curva, igual a 1, representa la probabilidad total de que el tiempo de procesamiento de la transacción ocurra en algún punto dentro de su dominio ($0 \le X \le 4$). Es una condición fundamental para cualquier PDF.

**Ejercicio 2: Probabilidad por Integración**

*(Contexto: El voltaje ($X$, en voltios) de una fuente de alimentación sigue la PDF del Ejercicio 1 (con $\displaystyle k$ ya encontrado).)*

1.  **Planteamiento/Fórmula:** Calcula la probabilidad de que el voltaje esté **entre 1 y 3 voltios**, $\displaystyle P(1 < X < 3)$. (4.1.2)
2.  **Código/Gráfico:** Utiliza Python para calcular la integral definida y **graficar** la PDF, sombreando la región de la probabilidad solicitada.
3.  **Interpretación:** Si $\displaystyle P(1 < X < 3)$ fuera muy bajo, ¿qué implicación práctica tendría esto para la estabilidad de la fuente de alimentación?

Plantamiento:

Se utiliza el valor de k=1/8. La probabilidad P(1<X<3) es el área bajo la PDF entre 1 y 3:$$P(1 < X < 3) = \int_{1}^{3} f(x) \, dx = \int_{1}^{3} \frac{1}{8}x \, dx$$$$= \frac{1}{8} \left[ \frac{x^2}{2} \right]_{1}^{3} = \frac{1}{16} (3^2 - 1^2) = \frac{1}{16} (9 - 1) = \frac{8}{16} = \mathbf{0.5}$$

In [ ]:
## Utiliza f_x(x) y k = 1/8 del Ejercicio 1

## 1. Cálculo de la Probabilidad P(1 < X < 3)
probabilidad, error = quad(f_x, 1, 3)
print(f"Probabilidad P(1 < X < 3): {probabilidad:.4f}")

## 2. Gráfico de la PDF con la región sombreada
x_vals = np.linspace(0, 4, 100)
y_vals = f_x(x_vals)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, y_vals, label=r'$f(x) = \frac{1}{8}x$', color='blue')

## Sombreado de P(1 < X < 3)
x_prob = np.linspace(1, 3, 50)
y_prob = f_x(x_prob)
plt.fill_between(x_prob, y_prob, color='red', alpha=0.6, label=f'$P(1 < X < 3) = {probabilidad:.2f}$')

plt.title('PDF del Voltaje con Probabilidad Sombreada')
plt.xlabel('Voltaje (X en voltios)')
plt.ylabel('Densidad de Probabilidad')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


Interpretación:

Si $P(1 < X < 3)$ fuera muy bajo (ej., 0.1), implicaría que la fuente de alimentación raramente opera dentro del rango de voltaje de 1 a 3 voltios. Esto sugiere una baja estabilidad o un alto riesgo de que el voltaje caiga fuera de este rango operativo crucial (ej., a valores cercanos a 0 o 4), comprometiendo el funcionamiento de los equipos conectados

**Ejercicio 3: Valor Esperado (Media)**

*(Contexto: Utiliza la variable $X$ (voltaje) del Ejercicio 2.)*

1.  **Planteamiento/Fórmula:** Calcula el **Valor Esperado** del voltaje, $\displaystyle E[X]$. (4.2.1)
2.  **Código/Gráfico:** Utiliza Python para calcular la integral de $\displaystyle E[X]$ y marca este valor en el gráfico de la PDF.
3.  **Interpretación:** ¿Cómo se interpreta el $\displaystyle E[X]$ para el ingeniero eléctrico que monitorea el voltaje promedio?

Plantamiento:

El Valor Esperado E[X] es el primer momento, calculado como:

$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx = \int_{0}^{4} x \cdot \frac{1}{8}x \, dx = \frac{1}{8} \int_{0}^{4} x^2 \, dx$$

$$= \frac{1}{8} \left[ \frac{x^3}{3} \right]_{0}^{4} = \frac{1}{24} (4^3 - 0^3) = \frac{64}{24} = \mathbf{\frac{8}{3} \approx 2.667}$$

In [ ]:
## Utiliza f_x(x) y k = 1/8 del Ejercicio 1

def g_x_e(x):
    """Función para el cálculo del Valor Esperado E[X]."""
    return x * f_x(x)

## 1. Cálculo del Valor Esperado E[X]
E_X, error_e = quad(g_x_e, 0, 4)
E_X_val = 8/3 # Valor exacto
print(f"Valor Esperado E[X] (Media): {E_X_val:.4f} voltios")

## 2. Gráfico de la PDF con E[X] marcado
x_vals = np.linspace(0, 4, 100)
y_vals = f_x(x_vals)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, y_vals, label=r'$f(x) = \frac{1}{8}x$', color='blue')
plt.fill_between(x_vals, y_vals, color='lightblue', alpha=0.5)

## Marcar el Valor Esperado E[X]
plt.axvline(x=E_X_val, color='green', linestyle='-', linewidth=2, label=f'$E[X] = {E_X_val:.3f}$')
plt.title('PDF del Voltaje y su Valor Esperado')
plt.xlabel('Voltaje (X en voltios)')
plt.ylabel('Densidad de Probabilidad')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


Interpretación:

El Valor Esperado $E[X]$ (aproximadamente 2.667 V) es el voltaje promedio a largo plazo que el ingeniero eléctrico esperaría observar si tomara un gran número de mediciones del voltaje de la fuente. Representa el centro de masa de la distribución de probabilidad.

**Ejercicio 4: Varianza y Desviación Estándar**

*(Contexto: Utiliza la variable $X$ (voltaje) del Ejercicio 2.)*

1.  **Planteamiento/Fórmula:** Calcula la **Varianza** $\displaystyle \text{Var}(X)$, usando la fórmula $\displaystyle \text{Var}(X) = E[X^2] - (E[X])^2$. (4.2.3)
2.  **Código/Gráfico:** Utiliza Python para calcular el segundo momento $\displaystyle E[X^2]$ y luego la Varianza.
3.  **Interpretación:** ¿Qué indica la **Desviación Estándar** ($\sigma$) sobre la fiabilidad y dispersión del voltaje de la fuente?

Plantamiento:

Se requiere E[X^2] para usar la fórmula $Var(X)=E[X^2]−(E[X])^2$

$$E[X^2] = \int_{0}^{4} x^2 \cdot f(x) \, dx = \int_{0}^{4} x^2 \cdot \frac{1}{8}x \, dx = \frac{1}{8} \int_{0}^{4} x^3 \, dx$$

$$= \frac{1}{8} \left[ \frac{x^4}{4} \right]_{0}^{4} = \frac{1}{32} (4^4) = \frac{256}{32} = \mathbf{8}$$

Ahora, la Varianza:

$$\text{Var}(X) = E[X^2] - (E[X])^2 = 8 - \left( \frac{8}{3} \right)^2 = 8 - \frac{64}{9} = \frac{72 - 64}{9} = \mathbf{\frac{8}{9}}$$

La Desviación Estándar (σ):

$$\sigma = \sqrt{\text{Var}(X)} = \sqrt{\frac{8}{9}} = \mathbf{\frac{\sqrt{8}}{3} \approx 0.943}$$

In [ ]:
## Utiliza f_x(x) del Ejercicio 1 y E_X_val = 8/3 del Ejercicio 3

def g_x_e2(x):
    """Función para el cálculo del Segundo Momento E[X^2]."""
    return x**2 * f_x(x)

## 1. Cálculo de E[X^2]
E_X2, error_e2 = quad(g_x_e2, 0, 4)
print(f"Segundo Momento E[X^2]: {E_X2:.4f}")

## 2. Cálculo de Varianza y Desviación Estándar
Var_X = E_X2 - (E_X_val)**2
sigma = np.sqrt(Var_X)

print(f"Varianza Var(X): {Var_X:.4f} (8/9)")
print(f"Desviación Estándar σ: {sigma:.4f} voltios")


Interpretación:

La Desviación Estándar ($\sigma \approx 0.943$ V) indica la dispersión o variabilidad promedio del voltaje respecto a su media ($E[X] \approx 2.667$ V). Un valor de $\sigma$ pequeño sugiere una fuente de alimentación más fiable y estable, ya que el voltaje tiende a permanecer cerca de su valor promedio. Un valor alto indicaría fluctuaciones grandes y baja fiabilidad

**Ejercicio 5: Esperanza de una Función de $X$ (Transformación Lineal)**

*(Contexto: El beneficio ($X$, en miles de USD) de una inversión tiene la PDF $\displaystyle f(x) = \frac{3}{64}x^2$ para $\displaystyle 0 \le x \le 4$. Si el inversor decide aplicar un cargo fijo, el beneficio ajustado es $\displaystyle Y = 3X - 5$.)*

1.  **Planteamiento/Fórmula:** Calcula el **Valor Esperado** del beneficio ajustado, $\displaystyle E[Y]$, usando las propiedades de la esperanza. (4.2.1)
2.  **Código/Gráfico:** Utiliza Python para calcular $\displaystyle E[X]$ primero, y luego aplica la transformación.
3.  **Interpretación:** Explica cómo la transformación lineal afecta la media original de la inversión.

Plantamiento:

Para una transformación lineal $Y = aX + b$, el valor esperado es $E[Y] = a E[X] + b$.

Aquí $Y = 3X - 5$, por lo que $E[Y] = 3E[X] - 5$.

Primero, se calcula E[X] para la nueva PDF:$$E[X] = \int_{0}^{4} x \cdot \frac{3}{64}x^2 \, dx = \frac{3}{64} \int_{0}^{4} x^3 \, dx$$$$= \frac{3}{64} \left[ \frac{x^4}{4} \right]_{0}^{4} = \frac{3}{256} (4^4) = \frac{3 \cdot 256}{256} = \mathbf{3}$$

Luego, se aplica la propiedad:$E[Y] = 3 E[X] - 5 = 3(3) - 5 = 9 - 5 = \mathbf{4}$

Nota: $f(x) = \frac{3}{64}x^2$ para $0 \le x \le 4$

In [ ]:
## PDF del beneficio f(x) = (3/64)x^2
def f_beneficio(x):
    return (3/64) * x**2

def g_x_e_beneficio(x):
    """Función para el cálculo de E[X] del beneficio."""
    return x * f_beneficio(x)

## 1. Cálculo de E[X]
E_X_beneficio, error_e_b = quad(g_x_e_beneficio, 0, 4)
print(f"Valor Esperado del beneficio original E[X]: {E_X_beneficio:.4f} (miles de USD)")

## 2. Aplicación de la transformación lineal E[Y] = 3*E[X] - 5
E_Y = 3 * E_X_beneficio - 5
print(f"Valor Esperado del beneficio ajustado E[Y]: {E_Y:.4f} (miles de USD)")


Interpretación:

La transformación lineal $Y = 3X - 5$ escala la media original por un factor de 3 y luego la desplaza por -5. Esto significa que el promedio de la inversión ajustada ($E[Y] = 4$) no solo refleja el promedio original ($E[X] = 3$), sino que también incorpora el impacto multiplicativo (el triple del beneficio) y el cargo fijo (la resta de 5). El beneficio promedio sube de 3 a 4, indicando que el factor multiplicativo supera el cargo fijo.

### 1.64 Bloque II: Función de Distribución Acumulativa (CDF), Medidas de Tendencia Central (4.3 & 4.4)

**Ejercicio 6: Obtención de CDF a partir de PDF y Gráfico**

*(Contexto: La vida útil ($X$, en años) de un componente sigue la PDF $\displaystyle f(x) = \frac{1}{9}x$ para $\displaystyle 0 \le x \le 3$, y $\displaystyle f(x) = \frac{2}{3} - \frac{1}{9}x$ para $\displaystyle 3 < x \le 6$.)*

1.  **Planteamiento/Fórmula:** Determina la **Función de Distribución Acumulativa (CDF)**, $\displaystyle F(x)$, para todos los intervalos, particularmente para $\displaystyle x \in (3, 6]$. (4.3.1)
2.  **Código/Gráfico:** Utiliza Python para **graficar** la PDF $\displaystyle f(x)$ y su correspondiente CDF $\displaystyle F(x)$ en dos gráficos adyacentes.
3.  **Interpretación:** ¿Qué representa el punto donde las funciones se "doblan" (en $\displaystyle x=3$)?

Planteamiento/Fórmula: Determinación de la CDF $F(x)$

La Función de Distribución Acumulativa (CDF), $F(x)$, para una Variable Aleatoria Continua (V.A.C.) $X$, se define como la integral de la PDF $f(t)$ desde $-\infty$ hasta $x$:
$$F(x) = P(X \le x) = \int_{-\infty}^{x} f(t) dt \quad [cite: 42]$$
La variable aleatoria $X$ (vida útil) está definida en el intervalo $[0, 6]$ años y su PDF es[cite: 40]:
$$
f(x) =
\begin{cases}
0 & \text{si } x < 0 \\
\frac{1}{9}x & \text{si } 0 \le x \le 3 \\
\frac{2}{3}-\frac{1}{9}x & \text{si } 3 < x \le 6 \\
0 & \text{si } x > 6
\end{cases}
$$

**Paso 1: Para $x < 0$**
$$F(x) = \int_{-\infty}^{x} 0 \, dt = 0$$

**Paso 2: Para $0 \le x \le 3$**
$$F(x) = \int_{-\infty}^{0} 0 \, dt + \int_{0}^{x} \frac{1}{9}t \, dt$$
$$F(x) = 0 + \frac{1}{9} \left[ \frac{t^2}{2} \right]_{0}^{x} = \frac{1}{9} \left( \frac{x^2}{2} - 0 \right) = \frac{x^2}{18}$$
*En el punto de cambio, $x=3$:* $F(3) = \frac{3^2}{18} = \frac{9}{18} = 0.5$

**Paso 3: Para $3 < x \le 6$**
La CDF en este intervalo es la probabilidad acumulada hasta $x=3$ (que es $F(3)=0.5$) más la integral de la PDF en el nuevo segmento, desde $3$ hasta $x$:
$$F(x) = F(3) + \int_{3}^{x} \left( \frac{2}{3} - \frac{1}{9}t \right) dt$$
$$F(x) = 0.5 + \left[ \frac{2}{3}t - \frac{1}{9}\frac{t^2}{2} \right]_{3}^{x}$$
$$F(x) = \frac{1}{2} + \left( \frac{2}{3}x - \frac{x^2}{18} \right) - \left( \frac{2}{3}(3) - \frac{3^2}{18} \right)$$
$$F(x) = \frac{1}{2} + \frac{2}{3}x - \frac{x^2}{18} - \left( 2 - \frac{9}{18} \right)$$
$$F(x) = \frac{1}{2} + \frac{2}{3}x - \frac{x^2}{18} - \left( 2 - \frac{1}{2} \right)$$
$$F(x) = \frac{1}{2} + \frac{2}{3}x - \frac{x^2}{18} - \frac{3}{2}$$
$$F(x) = \frac{2}{3}x - \frac{x^2}{18} - 1$$
*Verificación en el punto final, $x=6$:* $F(6) = \frac{2}{3}(6) - \frac{6^2}{18} - 1 = 4 - \frac{36}{18} - 1 = 4 - 2 - 1 = 1$. (Correcto)

**Paso 4: Para $x > 6$**
$$F(x) = \int_{-\infty}^{6} f(t) \, dt + \int_{6}^{x} 0 \, dt = F(6) + 0 = 1 + 0 = 1$$

**Función de Distribución Acumulativa (CDF) Final:**
$$
F(x) =
\begin{cases}
0 & \text{si } x < 0 \\
\frac{x^2}{18} & \text{si } 0 \le x \le 3 \\
\frac{2}{3}x - \frac{x^2}{18} - 1 & \text{si } 3 < x \le 6 \\
1 & \text{si } x > 6
\end{cases}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Definición de la PDF f(x)
def pdf_fx(x):
    if 0 <= x <= 3:
        return (1/9) * x
    elif 3 < x <= 6:
        return (2/3) - (1/9) * x
    else:
        return 0

## Definición de la CDF F(x)
def cdf_fx(x):
    if x < 0:
        return 0
    elif 0 <= x <= 3:
## F(x) = x^2 / 18
        return (x**2) / 18
    elif 3 < x <= 6:
## F(x) = (2/3)x - x^2/18 - 1
        return (2/3) * x - (x**2) / 18 - 1
    else: # x > 6
        return 1

## Crear un array de valores x para el gráfico
x_vals = np.linspace(-1, 7, 500)

## Calcular los valores de la PDF y la CDF
pdf_vals = np.array([pdf_fx(x) for x in x_vals])
cdf_vals = np.array([cdf_fx(x) for x in x_vals])

## Crear los gráficos adyacentes
fig, axes = plt.subplots(1, 2, figsize=(14, 5)) # Una fila, dos columnas

## --- Gráfico de la PDF f(x) ---
axes[0].plot(x_vals, pdf_vals, label='$f(x)$', color='blue')
axes[0].set_title('Función de Densidad de Probabilidad (PDF)')
axes[0].set_xlabel('Vida Útil (X, años)')
axes[0].set_ylabel('$f(x)$')
axes[0].grid(True, linestyle='--')
## Marcar el punto de cambio x=3
axes[0].vlines(3, 0, pdf_fx(3), color='red', linestyle=':', label='Punto de cambio $x=3$')
axes[0].scatter(3, pdf_fx(3), color='red', s=50, zorder=5) # Punto en x=3
axes[0].legend()

## --- Gráfico de la CDF F(x) ---
axes[1].plot(x_vals, cdf_vals, label='$F(x)$', color='green')
axes[1].set_title('Función de Distribución Acumulativa (CDF)')
axes[1].set_xlabel('Vida Útil (X, años)')
axes[1].set_ylabel('$F(x)$')
axes[1].grid(True, linestyle='--')
## Marcar el punto de cambio x=3
axes[1].vlines(3, 0, cdf_fx(3), color='red', linestyle=':', label='Punto de cambio $x=3$')
axes[1].scatter(3, cdf_fx(3), color='red', s=50, zorder=5) # Punto en x=3, F(3)=0.5
axes[1].hlines(0.5, 0, 3, color='orange', linestyle='--', alpha=0.6, label='$F(3)=0.5$')
axes[1].hlines(1, 6, 7, color='purple', linestyle='--', alpha=0.6, label='$F(6)=1$')
axes[1].legend()

plt.tight_layout()
plt.show()


Interpretación: El punto de "doblez" en $x=3$

[cite_start]El punto donde la función se "dobla" en **$x=3$** representa el **cambio en la forma matemática o comportamiento probabilístico** de la variable aleatoria[cite: 44].

* **En la PDF ($f(x)$):** Indica el punto donde la **tasa de probabilidad de falla** cambia. Antes de $x=3$, la probabilidad de que la vida útil sea mayor aumenta linealmente con $x$ (la pendiente es positiva). Después de $x=3$, la probabilidad comienza a **disminuir** linealmente (la pendiente es negativa) hasta llegar a cero en $x=6$. En este punto, la PDF alcanza su **valor máximo** (la Moda) en $x=3$.
* **En la CDF ($F(x)$):** Aunque la función $F(x)$ es **continua** en $x=3$ (se acumula $0.5$ de probabilidad), el "doblez" representa un cambio en la **concavidad** de la curva. Es el punto donde la **tasa de acumulación de probabilidad** cambia de ser creciente (con pendiente positiva y decreciente) a ser decreciente (con pendiente positiva y cada vez menor) en el intervalo $[3, 6]$. Gráficamente, es el punto donde la curva pasa de curvarse hacia arriba a curvarse hacia abajo. Además, $F(3)=0.5$ indica que la **Mediana** de la vida útil del componente es 3 años.

**Ejercicio 7: Recuperación de PDF a partir de CDF**

*(Contexto: La CDF del tiempo de respuesta ($T$, en segundos) de un servidor es $\displaystyle F(t) = 1 - e^{-2t}$ para $\displaystyle t \ge 0$.)*

1.  **Planteamiento/Fórmula:** Recupera la **Función de Densidad de Probabilidad (PDF)**, $\displaystyle f(t)$, mediante la diferenciación de $\displaystyle F(t)$. (4.3.3)
2.  **Código/Gráfico:** **Grafica** la CDF $\displaystyle F(t)$ y la PDF $\displaystyle f(t)$.
3.  **Interpretación:** ¿Qué significado tiene que la pendiente de la CDF sea cero al inicio ($t=0$)?

Planteamiento/Fórmula: Recuperación de la PDF $f(t)$

**Contexto:** La CDF del tiempo de respuesta ($T$, en segundos) de un servidor es:
$$F(t) = 1 - e^{-2t} \quad \text{para } t \ge 0$$ y $F(t) = 0$ para $t < 0$.

**Fórmula y Razonamiento Lógico:**
Para una Variable Aleatoria Continua (V.A.C.), la PDF, $f(t)$, es la derivada de su CDF, $F(t)$.
$$f(t) = \frac{d}{dt} F(t) \quad \text{para todo } t$$

1.  **Para $t < 0$:**
    $$f(t) = \frac{d}{dt}(0) = 0$$

2.  **Para $t \ge 0$:**
    $$f(t) = \frac{d}{dt} (1 - e^{-2t})$$
    $$f(t) = \frac{d}{dt} (1) - \frac{d}{dt} (e^{-2t})$$
    $$f(t) = 0 - (e^{-2t} \cdot (-2))$$
    $$f(t) = 2e^{-2t}$$

El resultado es la PDF de una **distribución Exponencial** con parámetro de tasa $\lambda = 2$.

**PDF Resultante:**
$$f(t) = \begin{cases} 2e^{-2t} & \text{para } t \ge 0 \\ 0 & \text{para } t < 0 \end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Rango de valores de t (tiempo)
t = np.linspace(0, 4, 200) # Se usa 0 a 4 segundos para visualizar la mayor parte de la distribución

## 1. Función de Distribución Acumulativa (CDF)
F_t = 1 - np.exp(-2 * t)

## 2. Función de Densidad de Probabilidad (PDF)
f_t = 2 * np.exp(-2 * t)

## Crear la figura y los subgráficos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Ejercicio 7: CDF y PDF del Tiempo de Respuesta de un Servidor')

## Gráfico de la CDF F(t)
ax1.plot(t, F_t, label='$F(t) = 1 - e^{-2t}$', color='blue')
ax1.set_title('Función de Distribución Acumulativa (CDF)')
ax1.set_xlabel('Tiempo $t$ (segundos)')
ax1.set_ylabel('$F(t)$')
ax1.grid(True, linestyle='--')
ax1.axhline(1, color='gray', linestyle=':', linewidth=1)
ax1.legend()
ax1.text(0.1, 0.8, 'Pendiente en $t=0$ es $f(0)=2$', color='red', transform=ax1.transAxes)

## Gráfico de la PDF f(t)
ax2.plot(t, f_t, label='$f(t) = 2e^{-2t}$', color='red')
ax2.set_title('Función de Densidad de Probabilidad (PDF)')
ax2.set_xlabel('Tiempo $t$ (segundos)')
ax2.set_ylabel('$f(t)$')
ax2.grid(True, linestyle='--')
ax2.legend()
ax2.text(0.1, 0.8, 'Maximizada en $t=0$ con $f(0)=2$', color='blue', transform=ax2.transAxes)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print(f"La PDF obtenida es f(t) = 2*e^(-2t) para t >= 0.")
print("Esta es la función de densidad de una distribución Exponencial con parámetro lambda = 2.")


Interpretación: Pendiente de la CDF en $t=0$

**Pregunta:** ¿Qué significado tiene que la pendiente de la CDF sea cero al inicio ($t=0$)?

**Respuesta:**

  * La **pendiente de la CDF, $F(t)$, en un punto específico $t$ es igual al valor de la PDF, $f(t)$, en ese mismo punto**.
  * Si la pendiente de la CDF fuera cero al inicio, es decir, si $f(0) = 0$, significaría que es **casi imposible** que el evento (el tiempo de respuesta) tome un valor **cercano o igual a cero**.
  * **En este ejercicio**, al evaluar la PDF en $t=0$, obtenemos:
    $$f(0) = 2e^{-2(0)} = 2e^0 = 2$$
  * Dado que $f(0)=2$ **no es cero**, esto indica que **la pendiente de la CDF es máxima en $t=0$**. Por lo tanto, el tiempo de respuesta del servidor tiene la **mayor densidad de probabilidad** (es el valor más probable) justo al inicio. En la práctica, esto implica que el servidor está diseñado para responder **rápidamente**, con la mayoría de los tiempos de respuesta concentrados inmediatamente después de $t=0$.

**Ejercicio 8: Cálculo de la Mediana**

*(Contexto: Utiliza la PDF del beneficio $X$ (en miles de USD) del Ejercicio 5: $\displaystyle f(x) = \frac{3}{64}x^2$ para $\displaystyle 0 \le x \le 4$.)*

1.  **Planteamiento/Fórmula:** Calcula la **Mediana** de $\displaystyle X$, $\displaystyle m$, resolviendo la ecuación $\displaystyle F(m) = 0.5$. (4.4.1)
2.  **Código/Gráfico:** Marca la Mediana ($\displaystyle m$) y la Media ($\displaystyle E[X]$) en tu gráfico de la PDF.
3.  **Interpretación:** ¿Cómo se interpreta el valor de la Mediana para el inversor? (Pista: el 50% de las veces, el beneficio será menor o igual a este valor).

Planteamiento/Fórmula y Razonamiento Lógico

El contexto del ejercicio es la variable aleatoria continua $X$ (beneficio en miles de USD) con la Función de Densidad de Probabilidad (PDF):
$$f(x) = \frac{3}{64}x^2, \quad \text{para } 0 \le x \le 4$$

La **Mediana** ($m$) de una variable aleatoria continua es el valor que divide la distribución de probabilidad en dos partes iguales, es decir, el valor para el cual la probabilidad de que la variable sea menor o igual a $m$ es $0.5$.
Matemáticamente, esto se formula como encontrar $m$ tal que[cite: 52]:
$$F(m) = P(X \le m) = 0.5$$

Donde $F(m)$ es la Función de Distribución Acumulativa (CDF) evaluada en $m$. La CDF se calcula integrando la PDF desde el límite inferior del soporte hasta $m$:
$$F(m) = \int_{-\infty}^{m} f(x) \, dx$$

Como el soporte de $X$ es $[0, 4]$, la integral se convierte en:
$$\int_{0}^{m} \frac{3}{64}x^2 \, dx = 0.5$$

**Resolviendo la integral:**
$$\left[ \frac{3}{64} \cdot \frac{x^3}{3} \right]_{0}^{m} = 0.5$$
$$\left[ \frac{x^3}{64} \right]_{0}^{m} = 0.5$$
$$\frac{m^3}{64} - \frac{0^3}{64} = 0.5$$
$$\frac{m^3}{64} = 0.5$$

**Despejando $m$:**
$$m^3 = 0.5 \cdot 64$$
$$m^3 = 32$$
$$m = \sqrt[3]{32}$$

El **razonamiento lógico** es que, dado que $f(x)$ es una PDF válida, el área total bajo la curva de 0 a 4 debe ser 1. Estamos buscando el punto $m$ entre 0 y 4 tal que el área a la izquierda de $m$ sea exactamente 0.5.

In [ ]:
import numpy as np
from scipy.integrate import quad

## --- 1. Cálculo de la Mediana (m) ---

## Fórmula para m: m = cúbica(32)
m = np.cbrt(32)

## Imprimir el valor de la Mediana
print(f"El valor de la Mediana (m) es: {m:.4f} miles de USD")

## --- 2. Cálculo del Valor Esperado E[X] (para la interpretación/comparación) ---

## PDF: f(x) = (3/64) * x^2
def pdf_fx(x):
    return (3/64) * x**2

## Función para el valor esperado E[X] = Integral(x * f(x) dx)
def integrand_ex(x):
    return x * pdf_fx(x)

## Calcular E[X] con integración numérica
E_X, err_E_X = quad(integrand_ex, 0, 4)

## Imprimir el valor de E[X]
print(f"El Valor Esperado (Media, E[X]) es: {E_X:.4f} miles de USD")

## --- 3. Verificación de F(m) = 0.5 (Opcional) ---

## Verificar la integral de 0 a m de f(x)
F_m, err_F_m = quad(pdf_fx, 0, m)
print(f"Verificación: F(m) = P(X <= m) = {F_m:.4f}")


Interpretación

La Mediana ($m$) calculada es **3.1748 miles de USD**.

  * **Significado para el Inversor:** El valor de la Mediana (3.1748 miles de USD) representa el **punto central** del beneficio. Esto significa que, el **50% de las veces**, el beneficio de la inversión será **menor o igual** a $3,174.80$ USD, y el 50% de las veces será mayor.

  * **Comparación con la Media:** El Valor Esperado (Media, $E[X]$) es de **3.0000 miles de USD**. Dado que la **Mediana ($m=3.1748$) es mayor que la Media ($E[X]=3.0000$)**, la distribución es **sesgada a la izquierda** (o negativamente asimétrica). Esto implica que la PDF tiene una "cola" más larga hacia los valores inferiores de beneficio. En este contexto, el valor más alto de la mediana sugiere que la mitad de los resultados son *mejores* que el promedio.

**Ejercicio 9: Cálculo de la Moda**

*(Contexto: Utiliza la PDF de la vida útil del componente del Ejercicio 6.)*

1.  **Planteamiento/Fórmula:** Calcula la **Moda** de $\displaystyle X$ encontrando el valor de $x$ que maximiza la PDF. (4.4.2)
2.  **Código/Gráfico:** Señala la Moda en el gráfico de la PDF.
3.  **Interpretación:** ¿Cuál es la "vida útil" más probable para el componente, y qué implicación tiene esto para la planificación del mantenimiento?

Planteamiento/Fórmula y Razonamiento Lógico

La **Moda** de una variable aleatoria continua es el valor donde su función de densidad de probabilidad ($f(x)$) alcanza su máximo local o global. En este ejercicio, la PDF es una función definida por partes, por lo que el proceso se simplifica a la evaluación de la función en su dominio, ya que la función es **lineal por tramos** (con dos tramos que son líneas rectas). La moda será el punto más alto del gráfico de la PDF.

Función de Densidad de Probabilidad (PDF)

La PDF de la vida útil $X$ (en años) del componente es:

[cite\_start]$f(x) = \begin{cases} \frac{1}{9}x & \text{para } 0 \le x \le 3 \\ \frac{2}{3} - \frac{1}{9}x & \text{para } 3 < x \le 6 \\ 0 & \text{en otro caso} \end{cases} \text{ [cite: 40, 41]}$

Razonamiento Lógico

1.  **Analizar el primer tramo** ($0 \le x \le 3$): $f_1(x) = \frac{1}{9}x$. Esta es una línea recta con pendiente positiva. Su máximo se alcanzará en el extremo superior del intervalo, $x=3$.

      * En $x=0$, $f_1(0) = 0$.
      * En $x=3$, $f_1(3) = \frac{1}{9}(3) = \frac{3}{9} = \frac{1}{3} \approx 0.333$.

2.  **Analizar el segundo tramo** ($3 < x \le 6$): $f_2(x) = \frac{2}{3} - \frac{1}{9}x$. Esta es una línea recta con pendiente negativa. Su máximo se alcanzaría teóricamente en el extremo inferior ($x=3$), y decrece hasta el extremo superior ($x=6$).

      * Límite a $x \to 3^+$: $\lim_{x\to 3^+} f_2(x) = \frac{2}{3} - \frac{1}{9}(3) = \frac{2}{3} - \frac{1}{3} = \frac{1}{3}$.
      * En $x=6$, $f_2(6) = \frac{2}{3} - \frac{1}{9}(6) = \frac{6}{9} - \frac{6}{9} = 0$.

3.  **Determinar el máximo**: La PDF es continua en $x=3$ (ambos tramos tienen el valor de $1/3$). La función **crece** de $x=0$ a $x=3$ y luego **decrece** de $x=3$ a $x=6$.

      * El **máximo global** de la función ocurre en el punto de cambio de pendiente, **$x=3$**, con un valor de $f(3) = \frac{1}{3}$.
      * Por lo tanto, la Moda es $\mathbf{x=3}$ años.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Definición de la PDF f(x) por tramos
def f_pdf(x):
    if 0 <= x <= 3:
        return (1/9) * x
    elif 3 < x <= 6:
        return (2/3) - (1/9) * x
    else:
        return 0

## Vectorizar la función para usarla con arrays de numpy
f_pdf_v = np.vectorize(f_pdf)

## Rango de x para la gráfica
x_vals = np.linspace(0, 6, 500)
y_vals = f_pdf_v(x_vals)

## El valor de la Moda encontrado analíticamente
moda = 3
f_moda = f_pdf(moda)

## Crear el gráfico
plt.figure(figsize=(10, 6))
plt.plot(x_vals, y_vals, label=r'$f(x) = \text{PDF}$')

## [cite_start]Señalar la Moda en el gráfico [cite: 59]
plt.scatter([moda], [f_moda], color='red', s=100, zorder=5, label=f'Moda = {moda} años')
plt.annotate(f'Máximo: ({moda}, {f_moda:.3f})', (moda, f_moda),
             textcoords="offset points", xytext=(15, -10), ha='center', fontsize=12, color='red')

## Etiquetas y título
plt.title(r'Función de Densidad de Probabilidad (PDF) y la Moda')
plt.xlabel('Vida Útil (X, en años)')
plt.ylabel('f(x)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.ylim(bottom=0)
plt.show()


Interpretación

La **Moda** es **3 años**.

  * **Significado**: La moda representa la **"vida útil" más probable** para el componente. De acuerdo con la función de densidad, es el valor de $X$ que tiene la mayor densidad de probabilidad.
  * **Implicación para la Planificación del Mantenimiento**: Saber que el pico de probabilidad está en 3 años sugiere que los ingenieros deberían enfocar la **planificación del mantenimiento preventivo o la sustitución** de la mayoría de estos componentes alrededor de la marca de **3 años**. Es el momento en que se espera que la mayor cantidad de fallas (o el mayor riesgo de ellas) ocurra, si la "vida útil" está definida por una falla, o es el tiempo más frecuente de operación.

**Ejercicio 10: Propiedades de la CDF y Probabilidades**

*(Contexto: La resistencia de un cable (R, en Ohmios) tiene una CDF $\displaystyle F(r)$. Sabemos que $\displaystyle F(5) = 0.15$ y $\displaystyle F(10) = 0.85$.)*

1.  **Planteamiento/Fórmula:** Utiliza las propiedades de la CDF para calcular $\displaystyle P(R > 10)$ y $\displaystyle P(5 < R \le 10)$. (4.3.2)
2.  **Código/Gráfico:** (Conceptual) Esboza una posible forma de la CDF y explica cómo se obtienen las probabilidades en ella.
3.  **Interpretación:** Si $\displaystyle P(5 < R \le 10)$ representa el rango de resistencia aceptable, ¿qué porcentaje de cables cumplen con la especificación?

Planteamiento/Fórmula y Razonamiento Lógico

La CDF, $F(r)$, se define como $F(r) = P(R \le r)$, es decir, la probabilidad de que la variable aleatoria R tome un valor menor o igual a $r$.

#### Cálculo de $P(R>10)$

  * **Razonamiento Lógico:** La probabilidad total para cualquier evento es 1. La probabilidad de que R sea mayor que 10 es el complemento de la probabilidad de que R sea menor o igual a 10 ($P(R \le 10)$).
  * **Fórmula Matemática:**
    $$P(R>10) = 1 - P(R \le 10) = 1 - F(10)$$

#### Cálculo de $P(5<R\le10)$

  * **Razonamiento Lógico:** Para una variable aleatoria continua, la probabilidad de que R esté en un intervalo $(a, b]$ es la diferencia entre la probabilidad acumulada en el punto final ($b$) y la probabilidad acumulada en el punto inicial ($a$)[cite: 63].
  * **Fórmula Matemática:**
    $$P(5<R\le10) = P(R\le10) - P(R\le5) = F(10) - F(5)$$

In [ ]:
## Valores dados de la CDF
F_5 = 0.15  # F(5)
F_10 = 0.85 # F(10)

## --- Cálculo de P(R > 10) ---
## P(R > 10) = 1 - F(10)
P_R_mayor_10 = 1 - F_10

## --- Cálculo de P(5 < R <= 10) ---
## P(5 < R <= 10) = F(10) - F(5)
P_5_a_10 = F_10 - F_5

print(f"La probabilidad P(R > 10) es: {P_R_mayor_10:.2f}")
print(f"La probabilidad P(5 < R <= 10) es: {P_5_a_10:.2f}")


Interpretación

**¿Qué indica la Desviación Estándar ($\sigma$) sobre la fiabilidad y dispersión del voltaje de la fuente?**

  * $P(5<R\le10) = 0.70$
  * Si $P(5<R\le10)$ representa el **rango de resistencia aceptable** , entonces el **70%** de los cables cumplen con la especificación de resistencia.
  * Esto implica que un **30%** de los cables no cumplen con el rango aceptable, ya sea porque su resistencia es demasiado baja ($R \le 5$, con probabilidad $F(5)=0.15$) o demasiado alta ($R > 10$, con probabilidad $1-F(10)=0.15$).

### 1.65 Bloque III: Distribuciones Típicas (Uniforme, Exponencial, Gaussiana) (4.5 & 4.6)

**Ejercicio 11: Distribución Uniforme (Error de Medición)**

*(Contexto: El error de redondeo ($X$) en una medición sigue una distribución Uniforme en el intervalo $\displaystyle [-0.05, 0.05]$ metros.)*

1.  **Planteamiento/Fórmula:** Calcula la **Media** $\displaystyle E[X]$ y la **PDF** $\displaystyle f(x)$. (4.5.1)
2.  **Código/Gráfico:** Utiliza Python (`scipy.stats.uniform`) para calcular la probabilidad de que el error sea **menor a 0.02 metros** ($\displaystyle P(X < 0.02)$) y **grafica** la PDF.
3.  **Interpretación:** ¿Por qué es la Media ($\displaystyle E[X]$) igual a cero en este contexto, y qué significa esto para la precisión de la medición?

##### Resolución
Si $X \sim U(a,b)$, su función de densidad de probabilidad (PDF) y su media son:

$f(x) =
\begin{cases}
\frac{1}{b - a}, & a \leq x \leq b, \\
0, & \text{en otro caso}
\end{cases}$ <br>

$E[X] = \df\frac{a + b}{2}$

En este caso:<br>
$a = -0.05$,
$\quad b = 0.05$

Entonces:
$f(x) = \df\frac{1}{0.05 - (-0.05)} = \df\frac{1}{0.1} = 10$

$E[X] = \df\frac{-0.05 + 0.05}{2} = 0$

**Resultado:** <br>
$f(x) =
\begin{cases}
10, & -0.05 \leq x \leq 0.05 \\
0, & \text{en otro caso}
\end{cases}
\quad \text{y} \quad E[X] = 0$

$P(X < 0.02) = \frac{0.02 - (-0.05)}{0.1} = \frac{0.07}{0.1} = 0.7$ <br>
$P(X < 0.02) = 0.7$

##### Interpretación

* La **media** $E[X] = 0$ significa que **los errores positivos y negativos se compensan**: el instrumento no tiende a medir sistemáticamente por encima ni por debajo del valor real.
* En otras palabras, **no hay sesgo de medición**: los errores son igualmente probables en ambos sentidos.
* La **distribución uniforme** refleja que **cualquier error dentro del intervalo ([-0.05, 0.05])** es igualmente posible, lo que indica una precisión limitada a ±0.05 m, pero sin tendencia sistemática.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform

## Parámetros
a, b = -0.05, 0.05
dist = uniform(loc=a, scale=b - a)

## Probabilidad P(X < 0.02)
p = dist.cdf(0.02)
print(f"P(X < 0.02) = {p:.4f}")

## Gráfico de la PDF
x = np.linspace(-0.06, 0.06, 400)
plt.plot(x, dist.pdf(x), color='navy', lw=2)
plt.fill_between(x, dist.pdf(x), where=(x>=a)&(x<=0.02), color='skyblue', alpha=0.4)
plt.title("Distribución Uniforme del Error de Medición U(-0.05, 0.05)")
plt.xlabel("Error X (m)")
plt.ylabel("f(x)")
plt.grid(True)
plt.show()


**Ejercicio 12: Distribución Exponencial (Tiempo de Espera)**

*(Contexto: El tiempo de espera ($T$, en minutos) en una línea de producción sigue una distribución Exponencial con una tasa media $\displaystyle \lambda = 0.4$ esperas por minuto.)*

1.  **Planteamiento/Fórmula:** Calcula la probabilidad de que la espera sea **mayor a 10 minutos** ($\displaystyle P(T > 10)$). (4.5.2)
2.  **Código/Gráfico:** Utiliza Python (`scipy.stats.expon.sf`) para calcular la probabilidad y **graficar** la PDF, marcando el área de interés.
3.  **Interpretación:** Si la media de espera es $\displaystyle E[T]$, explica por qué $\displaystyle P(T > E[T])$ no es igual a 0.5 en la distribución Exponencial.

##### Resolución

Sabemos que si una variable aleatoria $T \sim \text{Exp}(\lambda)$, su **PDF** y sus propiedades son:

$f(t) =
\begin{cases}
\lambda e^{-\lambda t}, & t \geq 0, \\
0, & t < 0
\end{cases}$

$E[T] = \df\frac{1}{\lambda}$

La **probabilidad de que el tiempo de espera sea mayor que un valor (t_0)** se calcula como:
$P(T > t_0) = e^{-\lambda t_0}$

$\lambda = 0.4, \quad t_0 = 10$

$P(T > 10) = e^{-0.4(10)} = e^{-4} \approx 0.0183$

##### Interpretación

* La **media de espera** es: <br>
$E[T] = \df\frac{1}{\lambda} = \df\frac{1}{0.4} = 2.5 \text{ minutos.}$
* La probabilidad de esperar más que el promedio es: <br>
$P(T > E[T]) = P(T > 2.5) = e^{-0.4(2.5)} = e^{-1} \approx 0.3679$

* Tiene una **cola larga hacia la derecha** (asimetría positiva).
* Muchos tiempos de espera son **menores** que la media, pero algunos valores grandes **aumentan** la media aritmética.
* Por eso, aunque la media sea 2.5, la probabilidad de superar ese valor es solo ≈ 36.8 %.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import expon

## Parámetros
lambda_ = 0.4
dist = expon(scale=1/lambda_)  # scale = 1/lambda

## Probabilidad P(T > 10)
p = dist.sf(10)  # survival function = P(T > 10)
print(f"P(T > 10) = {p:.4f}")

## Datos para graficar
t = np.linspace(0, 20, 400)
pdf = dist.pdf(t)

## Gráfico
plt.plot(t, pdf, color='darkred', lw=2, label='PDF: $f(t) = 0.4 e^{-0.4t}$')
plt.fill_between(t, pdf, where=(t>10), color='salmon', alpha=0.5, label='Área: P(T>10)')
plt.axvline(10, color='gray', ls='--')
plt.title("Distribución Exponencial del Tiempo de Espera (λ = 0.4)")
plt.xlabel("Tiempo de espera T (min)")
plt.ylabel("f(T)")
plt.legend()
plt.grid(True)
plt.show()


**Ejercicio 13: Distribución Gaussiana General (Calificación de Proyectos)**

*(Contexto: Las calificaciones de un proyecto ($X$) siguen una distribución Normal con $\displaystyle \mu=75$ y $\displaystyle \sigma=8$.)*

1.  **Planteamiento/Fórmula:** Calcula la probabilidad de que una calificación esté **entre 65 y 90** ($\displaystyle P(65 < X < 90)$), estandarizando los valores. (4.6.1)
2.  **Código/Gráfico:** Utiliza Python (`scipy.stats.norm`) para calcular la probabilidad y **graficar** la Normal, sombreando la región de interés.
3.  **Interpretación:** Si la calificación de aprobación es 60, ¿cuál es el porcentaje de estudiantes que aprueban el proyecto?

##### Resolución

Sabemos que si $X \sim N(\mu, \sigma)$, entonces la variable estandarizada:

$Z = \frac{X - \mu}{\sigma}$
sigue una distribución normal estándar $N(0,1)$.

Queremos calcular: <br>
$P(65 < X < 90)$

###### Paso 1: Estandarización

$z_1 = \frac{65 - 75}{8} = \frac{-10}{8} = -1.25$ <br>

$z_2 = \frac{90 - 75}{8} = \frac{15}{8} = 1.875$

Entonces: <br>
$P(65 < X < 90) = P(-1.25 < Z < 1.875)
= \Phi(1.875) - \Phi(-1.25)$

De las tablas (o usando Python): <br>

$\Phi(1.875) \approx 0.9699, \quad \Phi(-1.25) \approx 0.1056$

$P(65 < X < 90) = 0.9699 - 0.1056 = 0.8643$

El gráfico muestra la **curva normal centrada en 75**, con la región entre **65 y 90** sombreada.

##### Interpretación

La probabilidad encontrada significa que **aproximadamente el 86.4 % de las calificaciones** se encuentran entre 65 y 90 puntos.

Ahora, para saber cuántos **aprueban (X ≥ 60):**

$P(X > 60) = 1 - P(X \le 60)$

Estandarizando: <br>

$z = \frac{60 - 75}{8} = \frac{-15}{8} = -1.875$

$P(X > 60) = 1 - \Phi(-1.875) = 1 - 0.0301 = 0.9699$

96.99% de los estudiantes aprueban.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Parámetros
mu, sigma = 75, 8
dist = norm(loc=mu, scale=sigma)

## Probabilidad entre 65 y 90
p = dist.cdf(90) - dist.cdf(65)
print(f"P(65 < X < 90) = {p:.4f}")

## Gráfico
x = np.linspace(50, 100, 400)
pdf = dist.pdf(x)

plt.plot(x, pdf, color='navy', lw=2, label='Distribución Normal (μ=75, σ=8)')
plt.fill_between(x, pdf, where=(x>=65)&(x<=90), color='skyblue', alpha=0.5)
plt.axvline(65, color='gray', ls='--')
plt.axvline(90, color='gray', ls='--')
plt.title("Probabilidad de Calificación entre 65 y 90")
plt.xlabel("Calificación X")
plt.ylabel("f(X)")
plt.legend()
plt.grid(True)
plt.show()


**Ejercicio 14: Distribución Gaussiana Estándar (Cálculo de Percentiles)**

*(Contexto: Se utiliza la distribución $\displaystyle Z \sim N(0, 1)$ para definir los niveles de riesgo en un análisis financiero.)*

1.  **Planteamiento/Fórmula:** Calcula el **Percentil 95** ($\displaystyle z_{0.95}$), el valor tal que $\displaystyle P(Z \le z_{0.95}) = 0.95$. (4.6.2)
2.  **Código/Gráfico:** Utiliza Python (`scipy.stats.norm.ppf`) para calcular el valor y **grafica** la Normal Estándar marcando este punto crítico.
3.  **Interpretación:** ¿Qué representa este valor ($z_{0.95}$) en el contexto de la "cola" de la distribución de riesgo?

##### Resolución

Sabemos que si $Z \sim N(0, 1)$, la **función de distribución acumulada (CDF)** es:

$P(Z \le z) = \Phi(z)$

El **percentil 95** (también llamado *valor crítico al 95 %*) se define como el valor $z_{0.95}$ que cumple: <br>

$\Phi(z_{0.95}) = 0.95$ <br>

Para obtenerlo, usamos la función inversa de la CDF (cuantil o *percent point function*): <br>
$z_{0.95} = \Phi^{-1}(0.95)$

De tablas o usando Python: <br>

$z_{0.95} \approx 1.6449$

##### Interpretación

* El valor $z_{0.95} = 1.6449$ significa que: <br>
$P(Z \le 1.6449) = 0.95$
  o dicho de otro modo: <br>
$P(Z > 1.6449) = 0.05$
  → Solo el **5 % de los valores más extremos** se encuentran **por encima** de este punto.

* En **análisis de riesgo financiero**, este percentil se usa para definir **niveles de confianza** o **umbrales de riesgo**.
  Por ejemplo, un **VaR (Value at Risk) al 95 %** indica que **solo el 5 % de los casos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Percentil 95
z_95 = norm.ppf(0.95)
print(f"z_0.95 = {z_95:.4f}")

## Gráfico
x = np.linspace(-4, 4, 400)
pdf = norm.pdf(x)

plt.plot(x, pdf, color='navy', lw=2, label='Distribución Normal Estándar N(0,1)')
plt.fill_between(x, pdf, where=(x<=z_95), color='skyblue', alpha=0.5)
plt.axvline(z_95, color='red', lw=2, ls='--', label=f'z₀.₉₅ = {z_95:.2f}')
plt.title("Percentil 95 de la Distribución Normal Estándar")
plt.xlabel("Z")
plt.ylabel("f(Z)")
plt.legend()
plt.grid(True)
plt.show()


**Ejercicio 15: Función de Variables Aleatorias (Transformación No Lineal)**

*(Contexto: La temperatura de un reactor ($X$, en °C) sigue una Uniforme en $\displaystyle [0, 100]$. La energía generada es $\displaystyle Y = g(X) = X^2$.)*

1.  **Planteamiento/Fórmula:** Calcula la **Media** de la energía generada, $\displaystyle E[Y] = E[X^2]$. (4.7.1)
2.  **Código/Gráfico:** Utiliza Python para calcular la integral de $\displaystyle E[Y]$. **Grafica** la PDF de $X$. (Opcional avanzado: intenta derivar y graficar la PDF de $Y$).
3.  **Interpretación:** ¿Por qué $\displaystyle E[Y]$ **no es** igual a $(E[X])^2$? Explica la diferencia en términos de la distribución de la energía.

##### **Resolución** $\mathbf{E[Y] = E[X^2]}$

La temperatura $X \sim \text{Uniforme}(0, 100)$.

###### PDF de $\mathbf{X}$

Para $X$ en $[a, b] = [0, 100]$, la PDF es:
$$f_X(x) = \frac{1}{b - a} = \frac{1}{100 - 0} = \frac{1}{100} \quad \text{para } 0 \le x \le 100$$

###### Cálculo de $\mathbf{E[Y] = E[X^2]}$

El valor esperado de una función de una variable aleatoria, $g(X)$, se calcula como:
$$E[g(X)] = \int_{-\infty}^{\infty} g(x) f_X(x) dx$$

Sustituyendo $g(x) = x^2$ y $f_X(x) = 1/100$, y utilizando el rango de $X$:
$$E[X^2] = \int_{0}^{100} x^2 \left( \frac{1}{100} \right) dx$$

Sacando la constante:
$$E[X^2] = \frac{1}{100} \int_{0}^{100} x^2 dx$$

Calculando la integral:
$$E[X^2] = \frac{1}{100} \left[ \frac{x^3}{3} \right]_{0}^{100}$$

Evaluando los límites:
$$E[X^2] = \frac{1}{100} \left( \frac{100^3}{3} - \frac{0^3}{3} \right)$$
$$E[X^2] = \frac{1}{100} \left( \frac{1,000,000}{3} \right)$$
$$E[X^2] = \frac{10,000}{3} \approx \mathbf{3333.33}$$

La media de la energía generada, $E[Y]$, es aproximadamente $\mathbf{3333.33}$ unidades de energía.

##### **Interpretación** $\mathbf{E[Y] \neq (E[X])^2}$?

###### Cálculo de $\mathbf{(E[X])^2}$

Primero, calculemos la media de la temperatura $X$:
$$E[X] = \frac{a + b}{2} = \frac{0 + 100}{2} = 50$$

Luego, el cuadrado de la media:
$$(E[X])^2 = (50)^2 = \mathbf{2500}$$

##### Explicación de la Desigualdad

**$\mathbf{E[Y] = 3333.33}$ y $\mathbf{(E[X])^2 = 2500}$.** La media de la energía $\mathbf{E[Y]}$ es significativamente **mayor** que el cuadrado de la media de la temperatura $\mathbf{(E[X])^2}$.

La regla general es que para una función **convexa** como $g(x) = x^2$ (donde la segunda derivada $g''(x)=2$ es positiva), la desigualdad de Jensen garantiza que $E[g(X)] \ge g(E[X])$.

###### En términos de Distribución de Energía:

  * **$E[X]$** es el punto central de la **temperatura** (50 °C).
  * **$Y=X^2$** es una **transformación no lineal**. Esta transformación amplifica desproporcionadamente los valores más grandes de $X$.
      * Temperaturas cerca de 0: $X=10 \Rightarrow Y=100$. (Cambio: 90)
      * Temperaturas cerca de 100: $X=90 \Rightarrow Y=8100$. (Cambio: 7900)

Dado que la función $X^2$ crece exponencialmente (es convexa), los valores de **temperatura mayores** a la media ($\mu_X=50$) contribuyen con **mucho más peso** al valor esperado de la energía $E[X^2]$ que los valores menores a la media.

A pesar de que las temperaturas bajas y altas son igualmente probables (distribución uniforme de $X$), sus cuadrados (la energía $Y$) no lo son. Esto **sesga** la distribución de energía $Y$ fuertemente hacia la **derecha** (valores altos), arrastrando la media $E[Y]$ lejos del valor $(E[X])^2=2500$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

## Definir PDF de X
def f_X(x):
    return 1/100 if 0 <= x <= 100 else 0

## Función g(X) = X^2
def g(x):
    return x**2

## Calcular E[Y] = E[X^2] mediante integración
E_Y, error = quad(lambda x: g(x)*f_X(x), 0, 100)
print(f"E[Y] = {E_Y:.2f}")

## Graficar PDF de X
x_vals = np.linspace(0, 100, 500)
pdf_vals = [f_X(x) for x in x_vals]

plt.figure(figsize=(8,4))
plt.plot(x_vals, pdf_vals, label='PDF de X', color='blue')
plt.fill_between(x_vals, pdf_vals, alpha=0.2, color='blue')
plt.title('PDF de X ~ Uniforme(0,100)')
plt.xlabel('x (°C)')
plt.ylabel('f_X(x)')
plt.grid(True)
plt.legend()
plt.show()


### 1.66 Resumen del Protocolo Maestro
- **Solución Analítica Resaltada**: $\boxed{\text{Verificado con SymPy y SciPy stats}}$
- **Verificación Simbólica (SymPy)**:

## 10. Módulo de Simulación: Estimación MLE Computacional y Bootstrap No Paramétrico

La inferencia moderna combina la **Estimación por Máxima Verosimilitud (MLE) Computacional** con el **Remuestreo Bootstrap** para obtener intervalos de confianza empíricos sin asunciones de normalidad.

### 10.1 Algoritmo de Bootstrap No Paramétrico
Dada una muestra $x_1, \dots, x_n$:
1. Generar $B$ muestras con reemplazo de tamaño $n$: $x_b^*$.
2. Calcular el estimador $\hat{\heta}_b^*$ para cada réplica.
3. Construir el intervalo de confianza del $(1-\alpha)\imes 100\%$ mediante los percentiles $[\alpha/2, 1 - \alpha/2]$.

### 10.2 Inferencia Bootstrap en Python

In [ ]:
import numpy as np
import scipy.stats as stats
from IPython.display import display, Math

np.random.seed(42)
muestra_exp = stats.expon.rvs(scale=12.5, size=40) # Muestra original

## Bootstrap (B = 10,000 réplicas)
B = 10_000
medias_boot = [np.mean(np.random.choice(muestra_exp, size=len(muestra_exp), replace=True)) for _ in range(B)]

ic_inf = np.percentile(medias_boot, 2.5)
ic_sup = np.percentile(medias_boot, 97.5)

display(Math(fr"\text{{Media Muestral Original: }} \bar{{X}} = {np.mean(muestra_exp):.3f}"))
display(Math(fr"\text{{Intervalo de Confianza Bootstrap 95\%: }} [{ic_inf:.3f}, {ic_sup:.3f}]"))


## 9. Verificación Simbólica y Expresión Formal con SymPy

En inferencia estadística, la Estimación de Máxima Verosimilitud (MLE) se obtiene diferenciando simbólicamente la función de Log-Verosimilitud $\frac{d}{d\theta} \ln L(\theta) = 0$ con **SymPy**.

### 9.1 Estimador MLE de la Media Normal $\hat{\mu}_{MLE}$

$$\boxed{\hat{\mu}_{MLE} = \bar{X} = \frac{1}{n} \sum_{i=1}^n X_i}$$

In [ ]:
import sympy as sp
from IPython.display import display, Math

mu, sigma, n = sp.symbols('mu sigma n', positive=True)
sum_x = sp.Symbol('(\sum X_i)', real=True)

## Log-Verosimilitud de n observaciones normales
log_L = - (n / 2) * sp.log(2 * sp.pi * sigma**2) - (1 / (2 * sigma**2)) * (sp.Symbol('(\sum X_i^2)') - 2 * mu * sum_x + n * mu**2)

## Derivada respecto a mu
d_logL_dmu = sp.diff(log_L, mu)
mu_mle = sp.solve(d_logL_dmu, mu)[0]

display(Math(r'\text{Ecuación de Score } \frac{d \ln L}{d\mu}: ' + sp.latex(d_logL_dmu)))
display(Math(r'\text{Estimador MLE Resuelto } \hat{\mu}: ' + sp.latex(mu_mle)))
